# 04-1. 출시 후 LLM 분류 리뷰 전처리

이 코드은 `04_run_llm_postlaunch_analysis_v3.ipynb`에서 생성한 LLM 분류 결과를 읽어, 후속 단계인 **LLM 기반 패치·운영 전략 제안**에 사용할 근거 데이터를 만든다.

흐름은 출시 전 분석의 `03-1`과 맞춘다.

```text
04. 리뷰 LLM 분류
→ 04-1. 분류된 리뷰 전처리
→ 04-2. LLM 기반 패치·운영 전략 제안
```

주의할 점은 다음과 같다.

- 이 코드에서는 LLM을 호출하지 않는다.
- LLM이 분류한 감정, 이슈 태그, urgency 후보를 그대로 최종 우선 검토 수준로 사용하지 않는다.
- 패치·운영 우선 검토 수준은 **이슈 반복 수, 부정·혼합 리뷰 수, Steam 비추천 리뷰 수, 최근 리뷰 반복 여부, 짧은 플레이타임 부정 반응**을 기준으로 규칙 기반 계산한다.
- `High urgency`는 04번 리뷰 분류 단계에서 LLM이 판단한 시급도 후보이므로, **우선 검토 수준 계산에는 직접 사용하지 않고 보조 설명 지표로만 유지한다.**
- 04-2에서는 이 코드에서 계산한 `action_group_hint`, `rule_priority_hint`, `priority_rule_detail`, `priority_reason`을 고정 근거로 사용한다.


# 1. 기본 설정

In [188]:
# ============================================================
# 기본 라이브러리
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


In [ ]:
# ============================================================
# 프로젝트 경로 설정
# ============================================================
ROOT = Path.cwd()

if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# ============================================================
# 출시 후 분석 대상 게임 설정
# ============================================================
# TARGET_GAME_KEY만 바꿔가면서 04 -> 04-1을 게임별로 실행하면 된다.

POSTLAUNCH_TARGET_GAMES = {
    "heroes_of_hammerwatch_2": {
        "appid": 619820,
        "game_name": "Heroes of Hammerwatch II",
        "game_slug": "heroes_of_hammerwatch_2",
    },
    "necrosmith_2": {
        "appid": 2277320,
        "game_name": "Necrosmith 2",
        "game_slug": "necrosmith_2",
    },
    "children_of_the_sun": {
        "appid": 1309950,
        "game_name": "Children of the Sun",
        "game_slug": "children_of_the_sun",
    },
    "laundry_store_simulator": {
        "appid": 3150440,
        "game_name": "Laundry Store Simulator",
        "game_slug": "laundry_store_simulator",
    },
    "endoparasitic_2": {
        "appid": 2990640,
        "game_name": "Endoparasitic 2",
        "game_slug": "endoparasitic_2",

    },
}

# 여기만 바꿔가면서 실행
TARGET_GAME_KEY = "heroes_of_hammerwatch_2"
# TARGET_GAME_KEY = "necrosmith_2"
# TARGET_GAME_KEY = "children_of_the_sun"
# TARGET_GAME_KEY = "laundry_store_simulator"
# TARGET_GAME_KEY = "endoparasitic_2"



TARGET_GAME = POSTLAUNCH_TARGET_GAMES[TARGET_GAME_KEY]
TARGET_APPID = TARGET_GAME["appid"]
TARGET_GAME_NAME = TARGET_GAME["game_name"]
GAME_SLUG = TARGET_GAME["game_slug"]

# ============================================================
# 출시 후 게임별 데이터 폴더
# ============================================================
POSTLAUNCH_OUTPUT_DIR = ROOT / "data" / "outputs" / "postlaunch"
RUNS_DIR = POSTLAUNCH_OUTPUT_DIR / "runs"
RUN_DIR = RUNS_DIR / GAME_SLUG
RUN_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 04번 LLM 실행 산출물 호출 경로
# ============================================================
RESULT_CSV_PATH = RUN_DIR / "llm_review_analysis_result.csv"
ISSUE_TAG_FLAT_PATH = RUN_DIR / "llm_issue_tags_flat.csv"
LLM_INPUT_PATH = RUN_DIR / "llm_input_reviews.csv"

# ============================================================
# 04-1 전처리 산출물 저장 폴더
# ============================================================
POSTLAUNCH_PREPROCESS_DIR = RUN_DIR / "postlaunch_preprocess_data"
POSTLAUNCH_PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 04-1 전처리 산출 파일
# ============================================================
POSTLAUNCH_REVIEW_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_review_base.csv"
POSTLAUNCH_ISSUE_SUMMARY_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_issue_summary.csv"
POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_patch_ops_evidence_base.csv"
TABLEAU_POSTLAUNCH_SOURCE_PATH = POSTLAUNCH_PREPROCESS_DIR / "tableau_postlaunch_patch_ops_source.csv"

print("ROOT:", ROOT)
print("분석 대상:", TARGET_GAME_NAME)
print("TARGET_APPID:", TARGET_APPID)
print("GAME_SLUG:", GAME_SLUG)
print("RUN_DIR:", RUN_DIR)
print("04번 LLM 결과 CSV:", RESULT_CSV_PATH)
print("04번 이슈 태그 CSV:", ISSUE_TAG_FLAT_PATH)
print("04번 LLM 입력 CSV:", LLM_INPUT_PATH)
print("04-1 전처리 저장 폴더:", POSTLAUNCH_PREPROCESS_DIR)

print("RESULT_CSV_PATH exists:", RESULT_CSV_PATH.exists())
print("ISSUE_TAG_FLAT_PATH exists:", ISSUE_TAG_FLAT_PATH.exists())
print("LLM_INPUT_PATH exists:", LLM_INPUT_PATH.exists())

ROOT: c:\Users\joon5\Documents\github\steam-indie-game-analysis
분석 대상: Endoparasitic 2
TARGET_APPID: 2990640
GAME_SLUG: endoparasitic_2
RUN_DIR: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2
04번 LLM 결과 CSV: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\llm_review_analysis_result.csv
04번 이슈 태그 CSV: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\llm_issue_tags_flat.csv
04번 LLM 입력 CSV: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\llm_input_reviews.csv
04-1 전처리 저장 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\postlaunch_preprocess_data
RESULT_CSV_PATH exists: True
ISSUE_TAG_FLAT_PATH exists: True
LLM_INPUT_PATH exists: True


# 2. 데이터 불러오기

In [190]:
# ============================================================
# 04번 LLM 분류 결과 불러오기
# ============================================================

result_df = pd.read_csv(RESULT_CSV_PATH, dtype={"recommendationid": "string"})
issue_df = pd.read_csv(ISSUE_TAG_FLAT_PATH, dtype={"recommendationid": "string"})

print("리뷰 단위 LLM 결과:", result_df.shape)
print("이슈 태그 단위 결과:", issue_df.shape)

display(result_df.head())
display(issue_df.head())


리뷰 단위 LLM 결과: (245, 20)
이슈 태그 단위 결과: (489, 15)


,analysis_status,recommendationid,appid,game_name,review_datetime,release_date,days_from_release,release_period,steam_label_text,playtime_at_review_hours,votes_up,weighted_vote_score,llm_sentiment,sentiment_score,llm_primary_issue,llm_issue_tags,llm_urgency_candidate,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation
0,success,176177053,2990640,Endoparasitic 2,2024-10-01 20:00:52,2024-10-01,0,D0-D30,positive,0.266667,6,0.510040,positive,5,positive_praise,"[{""category"": ""positive_praise"", ""sentiment"": ""positive"", ""evidence"": ""한 손으로 플레이 가능한 조작감과 훌륭한 속편이라는 점을 칭찬함""}]",low,"한 손으로 플레이 가능한 조작감이 훌륭하며, 전작을 잇는 뛰어난 속편임.",현재의 직관적인 조작 방식과 게임성을 유지 및 강화할 것.,exact_match
1,success,176177300,2990640,Endoparasitic 2,2024-10-01 20:05:29,2024-10-01,0,D0-D30,positive,0.383333,1,0.490937,positive,5,positive_praise,"[{""category"": ""positive_praise"", ""sentiment"": ""positive"", ""evidence"": ""게임이 매우 훌륭하다고 언급함""}]",low,게임을 플레이한 후 매우 훌륭하다는 첫인상을 남김.,초반 플레이 경험이 긍정적이므로 현재의 게임 디자인 방향을 유지할 것.,exact_match
2,success,176179490,2990640,Endoparasitic 2,2024-10-01 20:46:59,2024-10-01,0,D0-D30,positive,10.433333,34,0.749335,positive,4,content_volume,"[{""category"": ""content_volume"", ""sentiment"": ""neutral"", ""evidence"": ""적 유형과 무기 종류가 각각 3개뿐이라는 점을 지적함""}, {""category"": ""...",medium,"적과 무기 종류가 적은 점은 아쉽지만, 사운드트랙, 스토리, 개선된 전투 시스템 등 전반적인 게임성이 매우 뛰어남.","콘텐츠 볼륨(적/무기 종류) 확장을 고려하되, 현재의 긍정적인 사운드트랙과 스토리텔링 강점을 유지할 것.",exact_match
3,success,176179851,2990640,Endoparasitic 2,2024-10-01 20:54:30,2024-10-01,0,D0-D30,positive,1.150000,9,0.597647,positive,5,positive_praise,"[{""category"": ""positive_praise"", ""sentiment"": ""positive"", ""evidence"": ""Polished and bug free, fun additional gamepla...",low,"전작의 장점을 잘 계승한 완성도 높은 후속작으로, 버그 없이 쾌적한 플레이와 재미있는 전투 및 맵 디자인을 높게 평가함.","현재의 게임 플레이 요소와 맵 디자인의 완성도를 유지하며, 향후 업데이트에서도 안정적인 빌드 품질을 지속적으로 관리할 것.",exact_match
4,success,176181376,2990640,Endoparasitic 2,2024-10-01 21:27:57,2024-10-01,0,D0-D30,positive,2.066667,4,0.537572,positive,5,positive_praise,"[{""category"": ""gameplay_loop"", ""sentiment"": ""positive"", ""evidence"": ""stuck in a corner, out of ammo and low on healt...",low,탄약 부족과 체력 고갈 등 극한의 상황에서 오는 긴장감을 긍정적으로 평가하며 게임의 재미를 극찬함.,현재의 긴장감 넘치는 게임 플레이 루프와 난이도 밸런스를 유지하여 플레이어에게 몰입감 있는 경험을 지속적으로 제공할 것.,exact_match


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,llm_primary_issue,llm_urgency_candidate,release_period,playtime_at_review_hours,votes_up,weighted_vote_score,llm_issue_category,issue_name_kor,llm_issue_sentiment,llm_issue_evidence
0,176177053,2990640,Endoparasitic 2,positive,positive,positive_praise,low,D0-D30,0.266667,6,0.510040,positive_praise,긍정 칭찬,positive,한 손으로 플레이 가능한 조작감과 훌륭한 속편이라는 점을 칭찬함
1,176177300,2990640,Endoparasitic 2,positive,positive,positive_praise,low,D0-D30,0.383333,1,0.490937,positive_praise,긍정 칭찬,positive,게임이 매우 훌륭하다고 언급함
2,176179490,2990640,Endoparasitic 2,positive,positive,content_volume,medium,D0-D30,10.433333,34,0.749335,content_volume,콘텐츠 분량,neutral,적 유형과 무기 종류가 각각 3개뿐이라는 점을 지적함
3,176179490,2990640,Endoparasitic 2,positive,positive,content_volume,medium,D0-D30,10.433333,34,0.749335,gameplay_loop,게임플레이 루프,positive,크래프팅 시스템이 단순하지만 잘 작동하고 재미있음
4,176179490,2990640,Endoparasitic 2,positive,positive,content_volume,medium,D0-D30,10.433333,34,0.749335,story,스토리,positive,스토리가 전작보다 깊이 있고 재미있음


In [191]:
# ============================================================
# 필수 컬럼 확인
# ============================================================

required_result_cols = [
    "recommendationid", "appid", "game_name",
    "review_datetime", "release_date", "days_from_release", "release_period",
    "steam_label_text", "playtime_at_review_hours",
    "llm_sentiment", "llm_primary_issue", "llm_urgency_candidate",
    "llm_review_summary", "llm_suggested_action",
]

required_issue_cols = [
    "recommendationid", "appid", "game_name",
    "steam_label_text", "llm_sentiment", "llm_urgency_candidate",
    "release_period", "playtime_at_review_hours",
    "llm_issue_category", "issue_name_kor", "llm_issue_sentiment", "llm_issue_evidence",
]

missing_result_cols = [col for col in required_result_cols if col not in result_df.columns]
missing_issue_cols = [col for col in required_issue_cols if col not in issue_df.columns]

if missing_result_cols:
    raise ValueError(f"리뷰 결과 파일에 필요한 컬럼이 없습니다: {missing_result_cols}")

if missing_issue_cols:
    raise ValueError(f"이슈 태그 파일에 필요한 컬럼이 없습니다: {missing_issue_cols}")

print("필수 컬럼 확인 완료")


필수 컬럼 확인 완료


# 3. 리뷰 단위 전처리

리뷰 1개를 1행으로 두고, 이후 이슈 집계에 필요한 기본 플래그를 만든다.


In [192]:
# ============================================================
# 공통 함수
# ============================================================

def safe_rate(numerator, denominator):
    """0으로 나누는 경우를 방지한 비율 계산 함수."""
    if denominator is None or pd.isna(denominator) or denominator == 0:
        return 0.0
    return round(float(numerator) / float(denominator), 4)


def classify_playtime_stage(hours):
    """리뷰 작성 시점 플레이타임을 출시 후 운영 해석용 구간으로 분류한다."""
    if pd.isna(hours):
        return "unknown"
    if hours < 1:
        return "0-1h"
    if hours < 5:
        return "1-5h"
    if hours < 20:
        return "5-20h"
    if hours < 50:
        return "20-50h"
    return "50h+"


def classify_recency_group(review_dt, max_dt):
    """분석 데이터 내 최신 리뷰일 기준 최근성 구간을 만든다."""
    if pd.isna(review_dt) or pd.isna(max_dt):
        return "unknown"

    diff_days = (max_dt - review_dt).days

    if diff_days <= 30:
        return "last_30d"
    if diff_days <= 60:
        return "31-60d"
    if diff_days <= 90:
        return "61-90d"
    return "older_90d"


def normalize_text_value(x):
    """결측/공백 텍스트를 안정적으로 처리한다."""
    if pd.isna(x):
        return ""
    return str(x).strip()


In [193]:
# ============================================================
# 리뷰 단위 기본 전처리
# ============================================================

review_base = result_df.copy()

# 날짜/숫자 타입 정리
for col in ["review_datetime", "release_date"]:
    review_base[col] = pd.to_datetime(review_base[col], errors="coerce")

numeric_cols = [
    "days_from_release", "playtime_at_review_hours", "votes_up",
    "weighted_vote_score", "sentiment_score",
]
for col in numeric_cols:
    if col in review_base.columns:
        review_base[col] = pd.to_numeric(review_base[col], errors="coerce")

# 문자열 정리
text_cols = [
    "recommendationid", "game_name", "release_period", "steam_label_text",
    "llm_sentiment", "llm_primary_issue", "llm_urgency_candidate",
    "llm_review_summary", "llm_suggested_action",
]
for col in text_cols:
    if col in review_base.columns:
        review_base[col] = review_base[col].apply(normalize_text_value)

# 분석 기준일
# recent_30d는 실제 오늘 기준 최근 30일이 아니라,
# 분석 데이터 내 최신 리뷰일을 기준으로 계산한 상대적 최근성 구간이다.
MAX_REVIEW_DATETIME = review_base["review_datetime"].max()

# 파생 구간/플래그
review_base["playtime_stage"] = review_base["playtime_at_review_hours"].apply(classify_playtime_stage)
review_base["review_recency_group"] = review_base["review_datetime"].apply(lambda x: classify_recency_group(x, MAX_REVIEW_DATETIME))

review_base["steam_negative_flag"] = review_base["steam_label_text"].eq("negative")
review_base["steam_positive_flag"] = review_base["steam_label_text"].eq("positive")
review_base["llm_negative_or_mixed_flag"] = review_base["llm_sentiment"].isin(["negative", "mixed"])
review_base["llm_positive_flag"] = review_base["llm_sentiment"].eq("positive")
review_base["high_urgency_flag"] = review_base["llm_urgency_candidate"].eq("high")
review_base["early_playtime_flag"] = review_base["playtime_stage"].isin(["0-1h", "1-5h"])
review_base["recent_30d_flag"] = review_base["review_recency_group"].eq("last_30d")

# 저장 컬럼 정리
review_base_cols = [
    "analysis_status",
    "recommendationid", "appid", "game_name",
    "review_datetime", "release_date", "days_from_release", "release_period", "review_recency_group",
    "steam_label_text", "steam_positive_flag", "steam_negative_flag",
    "playtime_at_review_hours", "playtime_stage", "early_playtime_flag", "recent_30d_flag",
    "votes_up", "weighted_vote_score",
    "llm_sentiment", "sentiment_score", "llm_negative_or_mixed_flag", "llm_positive_flag",
    "llm_primary_issue", "llm_urgency_candidate", "high_urgency_flag",
    "llm_review_summary", "llm_suggested_action", "steam_llm_sentiment_relation",
]
review_base_cols = [col for col in review_base_cols if col in review_base.columns]
review_base = review_base[review_base_cols].copy()

print("리뷰 단위 전처리 결과:", review_base.shape)
print("최신 리뷰일:", MAX_REVIEW_DATETIME)
display(review_base.head())


리뷰 단위 전처리 결과: (245, 28)
최신 리뷰일: 2026-04-10 23:16:46


,analysis_status,recommendationid,appid,game_name,review_datetime,release_date,days_from_release,release_period,review_recency_group,steam_label_text,steam_positive_flag,steam_negative_flag,playtime_at_review_hours,playtime_stage,early_playtime_flag,recent_30d_flag,votes_up,weighted_vote_score,llm_sentiment,sentiment_score,llm_negative_or_mixed_flag,llm_positive_flag,llm_primary_issue,llm_urgency_candidate,high_urgency_flag,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation
0,success,176177053,2990640,Endoparasitic 2,2024-10-01 20:00:52,2024-10-01,0,D0-D30,older_90d,positive,True,False,0.266667,0-1h,True,False,6,0.510040,positive,5,False,True,positive_praise,low,False,"한 손으로 플레이 가능한 조작감이 훌륭하며, 전작을 잇는 뛰어난 속편임.",현재의 직관적인 조작 방식과 게임성을 유지 및 강화할 것.,exact_match
1,success,176177300,2990640,Endoparasitic 2,2024-10-01 20:05:29,2024-10-01,0,D0-D30,older_90d,positive,True,False,0.383333,0-1h,True,False,1,0.490937,positive,5,False,True,positive_praise,low,False,게임을 플레이한 후 매우 훌륭하다는 첫인상을 남김.,초반 플레이 경험이 긍정적이므로 현재의 게임 디자인 방향을 유지할 것.,exact_match
2,success,176179490,2990640,Endoparasitic 2,2024-10-01 20:46:59,2024-10-01,0,D0-D30,older_90d,positive,True,False,10.433333,5-20h,False,False,34,0.749335,positive,4,False,True,content_volume,medium,False,"적과 무기 종류가 적은 점은 아쉽지만, 사운드트랙, 스토리, 개선된 전투 시스템 등 전반적인 게임성이 매우 뛰어남.","콘텐츠 볼륨(적/무기 종류) 확장을 고려하되, 현재의 긍정적인 사운드트랙과 스토리텔링 강점을 유지할 것.",exact_match
3,success,176179851,2990640,Endoparasitic 2,2024-10-01 20:54:30,2024-10-01,0,D0-D30,older_90d,positive,True,False,1.150000,1-5h,True,False,9,0.597647,positive,5,False,True,positive_praise,low,False,"전작의 장점을 잘 계승한 완성도 높은 후속작으로, 버그 없이 쾌적한 플레이와 재미있는 전투 및 맵 디자인을 높게 평가함.","현재의 게임 플레이 요소와 맵 디자인의 완성도를 유지하며, 향후 업데이트에서도 안정적인 빌드 품질을 지속적으로 관리할 것.",exact_match
4,success,176181376,2990640,Endoparasitic 2,2024-10-01 21:27:57,2024-10-01,0,D0-D30,older_90d,positive,True,False,2.066667,1-5h,True,False,4,0.537572,positive,5,False,True,positive_praise,low,False,탄약 부족과 체력 고갈 등 극한의 상황에서 오는 긴장감을 긍정적으로 평가하며 게임의 재미를 극찬함.,현재의 긴장감 넘치는 게임 플레이 루프와 난이도 밸런스를 유지하여 플레이어에게 몰입감 있는 경험을 지속적으로 제공할 것.,exact_match


# 4. 이슈 태그 단위 전처리

리뷰 안의 여러 이슈 태그를 펼친 데이터를 정리한다.  
한 리뷰가 같은 이슈를 여러 번 가진 경우, 이슈 집계에서는 `recommendationid + issue_category` 기준으로 중복을 제거한다.


In [194]:
# ============================================================
# 이슈 태그 단위 기본 전처리
# ============================================================

issue_base = issue_df.copy()

# 날짜/최근성/리뷰 요약 등은 리뷰 단위 결과에서 가져온다.
review_merge_cols = [
    "recommendationid", "review_datetime", "release_date", "days_from_release",
    "review_recency_group", "playtime_stage", "steam_negative_flag", "steam_positive_flag",
    "llm_negative_or_mixed_flag", "llm_positive_flag", "high_urgency_flag",
    "early_playtime_flag", "recent_30d_flag",
    "llm_review_summary", "llm_suggested_action", "steam_llm_sentiment_relation",
]
review_merge_cols = [col for col in review_merge_cols if col in review_base.columns]

issue_base = issue_base.merge(
    review_base[review_merge_cols].drop_duplicates("recommendationid"),
    on="recommendationid",
    how="left",
    suffixes=("", "_review"),
)

# 숫자 타입 정리
for col in ["playtime_at_review_hours", "votes_up", "weighted_vote_score", "days_from_release"]:
    if col in issue_base.columns:
        issue_base[col] = pd.to_numeric(issue_base[col], errors="coerce")

# 문자열 정리
for col in [
    "recommendationid", "game_name", "steam_label_text", "llm_sentiment",
    "llm_primary_issue", "llm_urgency_candidate", "release_period",
    "llm_issue_category", "issue_name_kor", "llm_issue_sentiment", "llm_issue_evidence",
]:
    if col in issue_base.columns:
        issue_base[col] = issue_base[col].apply(normalize_text_value)

# 이슈 태그 기준 감정 플래그
issue_base["issue_positive_flag"] = issue_base["llm_issue_sentiment"].eq("positive")
issue_base["issue_negative_or_mixed_flag"] = issue_base["llm_issue_sentiment"].isin(["negative", "mixed"])
issue_base["issue_negative_flag"] = issue_base["llm_issue_sentiment"].eq("negative")
issue_base["issue_mixed_flag"] = issue_base["llm_issue_sentiment"].eq("mixed")

# 같은 리뷰 안에서 같은 이슈가 중복으로 들어온 경우 제거
issue_review_base = issue_base.drop_duplicates(["recommendationid", "llm_issue_category"]).copy()

print("이슈 태그 원본 행 수:", len(issue_base))
print("리뷰-이슈 중복 제거 후 행 수:", len(issue_review_base))
display(issue_review_base.head())


이슈 태그 원본 행 수: 489
리뷰-이슈 중복 제거 후 행 수: 477


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,llm_primary_issue,llm_urgency_candidate,release_period,playtime_at_review_hours,votes_up,weighted_vote_score,llm_issue_category,issue_name_kor,llm_issue_sentiment,llm_issue_evidence,review_datetime,release_date,days_from_release,review_recency_group,playtime_stage,steam_negative_flag,steam_positive_flag,llm_negative_or_mixed_flag,llm_positive_flag,high_urgency_flag,early_playtime_flag,recent_30d_flag,llm_review_summary,llm_suggested_action,steam_llm_sentiment_relation,issue_positive_flag,issue_negative_or_mixed_flag,issue_negative_flag,issue_mixed_flag
0,176177053,2990640,Endoparasitic 2,positive,positive,positive_praise,low,D0-D30,0.266667,6,0.510040,positive_praise,긍정 칭찬,positive,한 손으로 플레이 가능한 조작감과 훌륭한 속편이라는 점을 칭찬함,2024-10-01 20:00:52,2024-10-01,0,older_90d,0-1h,False,True,False,True,False,True,False,"한 손으로 플레이 가능한 조작감이 훌륭하며, 전작을 잇는 뛰어난 속편임.",현재의 직관적인 조작 방식과 게임성을 유지 및 강화할 것.,exact_match,True,False,False,False
1,176177300,2990640,Endoparasitic 2,positive,positive,positive_praise,low,D0-D30,0.383333,1,0.490937,positive_praise,긍정 칭찬,positive,게임이 매우 훌륭하다고 언급함,2024-10-01 20:05:29,2024-10-01,0,older_90d,0-1h,False,True,False,True,False,True,False,게임을 플레이한 후 매우 훌륭하다는 첫인상을 남김.,초반 플레이 경험이 긍정적이므로 현재의 게임 디자인 방향을 유지할 것.,exact_match,True,False,False,False
2,176179490,2990640,Endoparasitic 2,positive,positive,content_volume,medium,D0-D30,10.433333,34,0.749335,content_volume,콘텐츠 분량,neutral,적 유형과 무기 종류가 각각 3개뿐이라는 점을 지적함,2024-10-01 20:46:59,2024-10-01,0,older_90d,5-20h,False,True,False,True,False,False,False,"적과 무기 종류가 적은 점은 아쉽지만, 사운드트랙, 스토리, 개선된 전투 시스템 등 전반적인 게임성이 매우 뛰어남.","콘텐츠 볼륨(적/무기 종류) 확장을 고려하되, 현재의 긍정적인 사운드트랙과 스토리텔링 강점을 유지할 것.",exact_match,False,False,False,False
3,176179490,2990640,Endoparasitic 2,positive,positive,content_volume,medium,D0-D30,10.433333,34,0.749335,gameplay_loop,게임플레이 루프,positive,크래프팅 시스템이 단순하지만 잘 작동하고 재미있음,2024-10-01 20:46:59,2024-10-01,0,older_90d,5-20h,False,True,False,True,False,False,False,"적과 무기 종류가 적은 점은 아쉽지만, 사운드트랙, 스토리, 개선된 전투 시스템 등 전반적인 게임성이 매우 뛰어남.","콘텐츠 볼륨(적/무기 종류) 확장을 고려하되, 현재의 긍정적인 사운드트랙과 스토리텔링 강점을 유지할 것.",exact_match,True,False,False,False
4,176179490,2990640,Endoparasitic 2,positive,positive,content_volume,medium,D0-D30,10.433333,34,0.749335,story,스토리,positive,스토리가 전작보다 깊이 있고 재미있음,2024-10-01 20:46:59,2024-10-01,0,older_90d,5-20h,False,True,False,True,False,False,False,"적과 무기 종류가 적은 점은 아쉽지만, 사운드트랙, 스토리, 개선된 전투 시스템 등 전반적인 게임성이 매우 뛰어남.","콘텐츠 볼륨(적/무기 종류) 확장을 고려하되, 현재의 긍정적인 사운드트랙과 스토리텔링 강점을 유지할 것.",exact_match,True,False,False,False


# 5. 게임 단위 기본 요약

분석 대상 게임의 현재 상태를 확인하기 위한 내부 요약값을 만든다.  
다만 이번 04-1 산출물에서는 별도 CSV로 저장하지 않고, 필요하면 코드 화면에서만 확인한다.

In [195]:
# ============================================================
# 게임 단위 기본 요약
# ============================================================

def make_game_base(review_df, issue_review_df):
    rows = []

    for (appid, game_name), g in review_df.groupby(["appid", "game_name"], dropna=False):
        issue_g = issue_review_df[issue_review_df["appid"] == appid]

        review_count = g["recommendationid"].nunique()
        steam_positive_count = int(g["steam_positive_flag"].sum())
        steam_negative_count = int(g["steam_negative_flag"].sum())
        llm_positive_count = int(g["llm_positive_flag"].sum())
        llm_negative_mixed_count = int(g["llm_negative_or_mixed_flag"].sum())
        high_urgency_count = int(g["high_urgency_flag"].sum())

        rows.append({
            "appid": appid,
            "game_name": game_name,
            "review_count": review_count,
            "issue_tag_count": len(issue_g),
            "steam_positive_review_count": steam_positive_count,
            "steam_negative_review_count": steam_negative_count,
            "steam_positive_rate": safe_rate(steam_positive_count, review_count),
            "llm_positive_review_count": llm_positive_count,
            "llm_negative_mixed_review_count": llm_negative_mixed_count,
            "llm_negative_mixed_rate": safe_rate(llm_negative_mixed_count, review_count),
            "high_urgency_review_count": high_urgency_count,
            "high_urgency_rate": safe_rate(high_urgency_count, review_count),
            "first_review_datetime": g["review_datetime"].min(),
            "last_review_datetime": g["review_datetime"].max(),
            "mean_playtime_at_review_hours": round(g["playtime_at_review_hours"].mean(), 2),
            "median_playtime_at_review_hours": round(g["playtime_at_review_hours"].median(), 2),
        })

    return pd.DataFrame(rows)


game_base = make_game_base(review_base, issue_review_base)

print("게임 단위 요약:", game_base.shape)
display(game_base)


게임 단위 요약: (1, 16)


,appid,game_name,review_count,issue_tag_count,steam_positive_review_count,steam_negative_review_count,steam_positive_rate,llm_positive_review_count,llm_negative_mixed_review_count,llm_negative_mixed_rate,high_urgency_review_count,high_urgency_rate,first_review_datetime,last_review_datetime,mean_playtime_at_review_hours,median_playtime_at_review_hours
0,2990640,Endoparasitic 2,245,477,198,47,0.8082,148,96,0.3918,31,0.1265,2024-10-01 20:00:52,2026-04-10 23:16:46,8.28,7.43


# 6. 이슈 단위 요약

패치·운영 전략 생성의 핵심이 되는 이슈별 반복성 근거를 만든다.

여기서의 `action_group_hint`, `rule_priority_hint`는 LLM 판단값이 아니라 **04-1에서 사전에 정한 데이터 기준으로 계산한 파생 컬럼**이다.

## 우선 검토 수준 계산 원칙

| 기준 | 의미 | 반영 방식 |
|---|---|---|
| 이슈 반복 수 | 같은 이슈가 몇 개 리뷰에서 반복되는지 | 핵심 기준 |
| 부정·혼합 맥락 | LLM이 해당 이슈를 부정/혼합 맥락으로 분류했는지 | 핵심 기준 |
| Steam 비추천 맥락 | Steam 라벨 기준 비추천 리뷰에서도 해당 이슈가 나타나는지 | 핵심 기준 |
| 최근성 | 분석 데이터 내 최신 리뷰일 기준 최근 30일에도 같은 문제가 반복되는지 | 현재 운영 판단 기준 |
| 짧은 플레이타임 | 초반 플레이타임에서 부정·혼합 이슈가 나타나는지 | 초기 이탈 가능성 참고 |
| High urgency | 04번 LLM 리뷰 분류에서 나온 시급도 후보 | **우선 검토 수준 계산에는 직접 사용하지 않고 보조 설명으로만 사용** |

따라서 `High urgency`가 높다는 이유만으로 `상` 우선 검토 수준을 부여하지 않는다.  
`상` 우선 검토 수준은 반복성, 부정·혼합 맥락, Steam 비추천 맥락, 최근성 중 핵심 근거가 함께 확인될 때만 부여한다.


참고:
- `negative_mixed_review_count`는 LLM이 이슈 태그 단위에서 부정 또는 혼합 맥락으로 분류한 리뷰 수이며, Steam 비추천 라벨과는 별개의 LLM 분류값이다.
- 우선 검토 수준 계산에서는 LLM의 부정·혼합 분류만 사용하지 않고, Steam 원본 비추천 라벨에서 해당 이슈가 함께 나타났는지도 확인한다.
- `recent_30d`는 실제 오늘 기준 최근 30일이 아니라, 분석 데이터 내 최신 리뷰일을 기준으로 계산한 상대적 최근성 구간이다.



## 수정 반영: 최종 '상' 제한

기존 절대 기준만 사용하면 리뷰 수가 많은 게임에서 여러 이슈가 동시에 `상`으로 분류될 수 있다.
따라서 본 버전에서는 먼저 절대 기준으로 `priority_candidate`를 만든 뒤, 최종 `rule_priority_hint`는 게임 내 상대 우선순위 보정을 적용한다.
최종 `상`은 전체 개선 대상 이슈의 30% 이내, 최대 6개로 제한한다.
절대 기준상 `상` 후보였지만 최종 상위 이슈에 들지 못한 항목은 `중`으로 조정한다.


In [196]:
# ============================================================
# 대응 구분/우선 검토 힌트 함수
# ============================================================
# 튜터님 피드백 반영:
# 패치·운영 우선순위는 LLM이 직접 판단하지 않는다.
# 이 셀에서 사전에 정한 데이터 기준으로 action_group_hint와 rule_priority_hint를 계산한다.
#
# 수정 포인트:
# - 기존 절대 기준만 사용하면 리뷰 수가 많은 게임에서 대부분 '상'으로 분류될 수 있다.
# - 따라서 절대 기준으로 먼저 priority_candidate를 만들고,
#   최종 rule_priority_hint는 이번 게임 안에서 상대적으로 먼저 봐야 할 상위 이슈만 '상'으로 제한한다.
# - High urgency는 LLM 기반 보조 지표이므로 우선 검토 수준 계산에는 직접 사용하지 않는다.
#
# 핵심 기준:
# 1. 해당 이슈가 리뷰에서 반복적으로 나타났는가
# 2. 부정·혼합 맥락으로 자주 언급되었는가
# 3. Steam 비추천 리뷰에서도 함께 나타났는가
# 4. 최근 30일에도 반복되는가
# 5. 이슈 성격상 즉시 확인이 필요한 기술/진행 방해 이슈인가

# ------------------------------------------------------------
# 실제 04번 LLM 결과의 llm_issue_category 값과 맞춘 이슈 그룹
# ------------------------------------------------------------
IMMEDIATE_ISSUES = {
    "crash",
    "save_progression",
    "save_progress",
    "bug",
    "performance",
    "optimization",
}

SHORT_TERM_ISSUES = {
    "gameplay_loop",
    "balance",
    "difficulty",
    "ui_ux",
    "control",
    "controls",
    "progression_grind",
}

LONG_TERM_ISSUES = {
    "content_volume",
    "content_amount",
    "story",
    "graphics_audio",
    "multiplayer",
    "monetization",
    "pricing",
    "price_value",
    "translation_localization",
}

STRENGTH_ISSUES = {"positive_praise"}
OPS_ISSUES = {"developer_communication"}

# ------------------------------------------------------------
# 절대 기준 기반 우선순위 후보 기준
# ------------------------------------------------------------
# get_rule_priority_hint는 최종 우선순위가 아니라 priority_candidate 후보값을 만드는 데 사용한다.
# 최종 rule_priority_hint는 apply_relative_priority에서 상대 순위 보정을 거쳐 결정한다.

HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS = 20
HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS = 10
HIGH_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS = 5
HIGH_PRIORITY_MIN_AFFECTED_REVIEWS = 50

MID_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS = 10
MID_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS = 5
MID_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS = 3
MID_PRIORITY_MIN_AFFECTED_REVIEWS = 20

# ------------------------------------------------------------
# 최종 '상' 개수 제한 기준
# ------------------------------------------------------------
# '상'은 단순히 문제가 있다는 뜻이 아니라,
# 이번 분석 대상 게임에서 먼저 확인해야 하는 상위 이슈라는 의미로 제한한다.
# - 최대 6개
# - 전체 개선 대상 이슈의 30% 이내
MAX_HIGH_ISSUES = 6
MAX_HIGH_RATE = 0.30

# 'other'는 원인이 명확하지 않으므로 수치가 커도 바로 '상'으로 올리지 않는다.
# positive_praise는 강점 유지 항목이므로 개선 우선순위 '상'에서 제외한다.
NO_HIGH_ISSUES = {
    "positive_praise",
    "other",
}

ACTION_GROUP_RANK = {
    "즉시 확인": 0,
    "단기 개선": 1,
    "운영 커뮤니케이션 개선": 2,
    "장기 검토": 3,
    "검토 필요": 4,
    "강점 유지": 5,
}

PATCH_OPS_NOTE_MAP = {
    "crash": "크래시 발생 조건과 로그를 우선 확인하고, 재현 가능한 오류부터 수정한다.",
    "save_progression": "저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다.",
    "save_progress": "저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다.",
    "bug": "반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다.",
    "performance": "프레임 저하, 로딩, 끊김 등 성능 문제를 환경별로 점검한다.",
    "optimization": "최적화 이슈가 특정 구간이나 사양에서 반복되는지 확인한다.",
    "gameplay_loop": "반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다.",
    "balance": "전투, 성장, 보상, 적 난이도의 불균형 지점을 조정한다.",
    "difficulty": "초반 진입 장벽과 후반 난이도 피로를 구분해 난이도 옵션 또는 안내를 보강한다.",
    "ui_ux": "메뉴, 인벤토리, 퀘스트 안내, 조작 안내 등 편의성 문제를 개선한다.",
    "control": "이동, 전투, 상호작용 조작의 반응성과 키 설정 편의성을 점검한다.",
    "controls": "이동, 전투, 상호작용 조작의 반응성과 키 설정 편의성을 점검한다.",
    "progression_grind": "반복 성장과 노가다 피로를 줄일 수 있는 보상/성장 속도 조정을 검토한다.",
    "content_volume": "콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.",
    "content_amount": "콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.",
    "story": "서사 전달, 퀘스트 흐름, 엔딩/분기 만족도를 장기 개선 후보로 검토한다.",
    "graphics_audio": "그래픽/사운드가 몰입을 방해하는지와 강점으로 작동하는지를 함께 확인한다.",
    "multiplayer": "멀티플레이 요구가 반복되는 경우 개발 범위와 수요를 장기 로드맵에서 검토한다.",
    "monetization": "가격, DLC, 과금 관련 불만이 반복되는지 확인하고 커뮤니케이션 방식을 점검한다.",
    "pricing": "가격 대비 만족도 불만이 반복되는지 확인하고 할인/번들/콘텐츠 가치 전달을 검토한다.",
    "price_value": "가격 대비 만족도 불만이 반복되는지 확인하고 할인/번들/콘텐츠 가치 전달을 검토한다.",
    "translation_localization": "번역/현지화 불만이 실제 이해도와 진행 경험에 영향을 주는지 검토한다.",
    "developer_communication": "패치 노트, 공지, 커뮤니티 응답 등 운영 커뮤니케이션을 점검한다.",
    "positive_praise": "긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.",
    "other": "세부 리뷰를 확인해 반복되는 하위 원인이 있는지 검토한다.",
}


def get_action_group_hint(row):
    """이슈 성격과 부정 맥락을 기준으로 대응 구분을 계산한다."""
    issue = row["llm_issue_category"]
    affected = int(row["affected_review_count"])
    positive = int(row["positive_review_count"])
    neg_mixed = int(row["negative_mixed_review_count"])
    steam_negative = int(row["steam_negative_review_count"])

    # 긍정 칭찬은 개선 우선순위가 아니라 유지할 강점으로 분리한다.
    if issue in STRENGTH_ISSUES and positive >= neg_mixed:
        return "강점 유지"

    # 즉시 확인은 크래시/저장/버그/성능/최적화처럼 플레이를 직접 방해할 수 있는 이슈로 제한한다.
    # High urgency만으로 즉시 확인으로 올리지 않는다.
    if issue in IMMEDIATE_ISSUES and neg_mixed >= 5 and steam_negative >= 3:
        return "즉시 확인"

    if issue in SHORT_TERM_ISSUES and neg_mixed >= 5:
        return "단기 개선"

    if issue in OPS_ISSUES and (neg_mixed >= 3 or steam_negative >= 3):
        return "운영 커뮤니케이션 개선"

    if issue in LONG_TERM_ISSUES and neg_mixed >= 5:
        return "장기 검토"

    if positive > neg_mixed and positive >= max(10, affected * 0.5):
        return "강점 유지"

    return "검토 필요"


def get_rule_priority_hint(row):
    """절대 기준 기반 우선 검토 후보값을 계산한다.

    주의:
    - 이 함수의 결과는 최종 rule_priority_hint가 아니라 priority_candidate로 사용한다.
    - 최종 rule_priority_hint는 apply_relative_priority에서 상대 순위 보정을 거친다.
    - High urgency는 LLM 기반 보조 지표이므로 이 함수의 계산식에 넣지 않는다.
    """
    action_group = row["action_group_hint"]
    affected = int(row["affected_review_count"])
    neg_mixed = int(row["negative_mixed_review_count"])
    steam_negative = int(row["steam_negative_review_count"])
    recent_negative = int(row["recent_30d_negative_mixed_review_count"])

    if action_group == "강점 유지":
        return "하"

    # 상 후보: 부정·혼합 반복 + Steam 비추천 맥락 + 최근성 또는 큰 반복 규모가 함께 확인되는 경우
    high_by_recent = (
        neg_mixed >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS
        and steam_negative >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS
        and recent_negative >= HIGH_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS
    )

    high_by_volume = (
        affected >= HIGH_PRIORITY_MIN_AFFECTED_REVIEWS
        and neg_mixed >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS
        and steam_negative >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS
    )

    # 즉시 확인 그룹은 기술/진행 차단 가능성이 있으므로 최근성 기준이 조금 약해도 상 후보로 둔다.
    high_by_blocking_issue = (
        action_group == "즉시 확인"
        and neg_mixed >= 15
        and steam_negative >= 10
    )

    if high_by_recent or high_by_volume or high_by_blocking_issue:
        return "상"

    # 중 후보: 부정·혼합 또는 Steam 비추천 맥락이 일정 수준 확인되는 경우
    mid_by_negative = (
        neg_mixed >= MID_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS
        and steam_negative >= MID_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS
    )

    mid_by_recent = (
        recent_negative >= MID_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS
        and steam_negative >= 1
    )

    mid_by_volume = (
        affected >= MID_PRIORITY_MIN_AFFECTED_REVIEWS
        and neg_mixed >= 5
    )

    if mid_by_negative or mid_by_recent or mid_by_volume:
        return "중"

    return "하"


def apply_relative_priority(issue_summary):
    """절대 기준 후보값을 바탕으로 최종 상/중/하를 상대적으로 보정한다.

    목적:
    - 리뷰 수가 많은 게임에서 대부분의 이슈가 '상'으로 나오는 문제를 줄인다.
    - '상'은 이번 게임에서 먼저 확인할 상위 이슈로 제한한다.
    - LLM의 High urgency는 직접 사용하지 않는다.
    """
    out = issue_summary.copy()

    # 1. 절대 기준 기반 후보값 생성
    if "priority_candidate" not in out.columns:
        out["priority_candidate"] = out.apply(get_rule_priority_hint, axis=1)

    # 2. 기본값은 하
    out["rule_priority_hint"] = "하"
    out["priority_selection_note"] = "절대 기준상 하 또는 개선 우선순위 근거가 상대적으로 약함"

    # 3. 개선 우선순위 산정 대상
    priority_target_mask = (
        out["action_group_hint"].ne("강점 유지")
        & ~out["llm_issue_category"].isin(STRENGTH_ISSUES)
    )

    target_count = int(priority_target_mask.sum())
    max_high_count = min(
        MAX_HIGH_ISSUES,
        max(1, int(np.ceil(target_count * MAX_HIGH_RATE)))
    )

    # 4. 기존 절대 기준에서 '상' 후보였던 이슈 중,
    #    기타/강점 항목은 최종 '상'에서 제외한다.
    high_candidate_mask = (
        priority_target_mask
        & out["priority_candidate"].eq("상")
        & ~out["llm_issue_category"].isin(NO_HIGH_ISSUES)
    )

    high_candidates = out[high_candidate_mask].copy()

    if len(high_candidates) > 0:
        high_candidates["action_group_rank"] = (
            high_candidates["action_group_hint"]
            .map(ACTION_GROUP_RANK)
            .fillna(9)
        )

        # 대응 긴급도가 높은 그룹을 먼저 보고,
        # 같은 그룹 안에서는 부정·혼합, Steam 비추천, 최근 반복, 전체 언급 규모가 큰 순서로 정렬한다.
        high_candidates = high_candidates.sort_values(
            [
                "action_group_rank",
                "negative_mixed_review_count",
                "steam_negative_review_count",
                "recent_30d_negative_mixed_review_count",
                "affected_review_count",
            ],
            ascending=[True, False, False, False, False],
        )

        high_index = high_candidates.head(max_high_count).index
        out.loc[high_index, "rule_priority_hint"] = "상"
        out.loc[high_index, "priority_selection_note"] = (
            f"절대 기준 '상' 후보 중 이번 게임 내 상대 우선순위 상위 {max_high_count}개로 유지"
        )

    # 5. 절대 기준에서는 상 후보였지만 최종 상위 이슈에 들지 못한 항목은 중으로 내린다.
    #    문제 근거가 없다는 뜻이 아니라, 이번 게임에서 최우선 확인 대상은 아니라는 의미다.
    demoted_high_mask = (
        priority_target_mask
        & out["priority_candidate"].eq("상")
        & out["rule_priority_hint"].ne("상")
    )
    out.loc[demoted_high_mask, "rule_priority_hint"] = "중"
    out.loc[demoted_high_mask, "priority_selection_note"] = (
        "절대 기준상 '상' 후보였으나, 최종 '상' 개수 제한 기준에 따라 '중'으로 조정"
    )

    # 6. 절대 기준 중 후보는 중으로 유지한다.
    mid_mask = (
        priority_target_mask
        & out["priority_candidate"].eq("중")
        & out["rule_priority_hint"].ne("상")
    )
    out.loc[mid_mask, "rule_priority_hint"] = "중"
    out.loc[mid_mask, "priority_selection_note"] = "절대 기준상 '중' 후보로 분류"

    # 7. 기타 이슈는 원인이 명확하지 않으므로 최종 상으로 올리지 않는다.
    other_mask = out["llm_issue_category"].eq("other")
    out.loc[other_mask & out["rule_priority_hint"].eq("상"), "rule_priority_hint"] = "중"
    out.loc[other_mask, "priority_selection_note"] = (
        "기타 이슈는 원인이 명확하지 않아 세부 리뷰 확인 대상으로 분리"
    )

    # 8. 강점 유지 항목은 개선 우선순위가 아니므로 하로 둔다.
    strength_mask = out["action_group_hint"].eq("강점 유지")
    out.loc[strength_mask, "rule_priority_hint"] = "하"
    out.loc[strength_mask, "priority_selection_note"] = "강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리"

    return out


def get_priority_rule_detail(row):
    """우선순위가 어떤 규칙 때문에 부여되었는지 설명용 라벨을 만든다."""
    priority = row["rule_priority_hint"]
    candidate = row.get("priority_candidate", priority)
    action_group = row["action_group_hint"]
    affected = int(row["affected_review_count"])
    neg_mixed = int(row["negative_mixed_review_count"])
    steam_negative = int(row["steam_negative_review_count"])
    recent_negative = int(row["recent_30d_negative_mixed_review_count"])

    if action_group == "강점 유지":
        return "강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리"

    if row["llm_issue_category"] == "other":
        return "기타 이슈는 원인 범주가 넓어 세부 리뷰 확인 대상으로 분리"

    if priority == "상":
        return "절대 기준 상 후보 + 게임 내 상대 우선순위 상위 이슈"

    if priority == "중" and candidate == "상":
        return "절대 기준 상 후보였으나 최종 상위 이슈 제한 기준에 따라 중으로 조정"

    if priority == "중":
        if neg_mixed >= MID_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS and steam_negative >= MID_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS:
            return "부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인"
        if recent_negative >= MID_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS:
            return "최근 30일 부정·혼합 반복이 일부 확인"
        if affected >= MID_PRIORITY_MIN_AFFECTED_REVIEWS:
            return "전체 반복 규모는 있으나 상 기준에는 미달"
        return "중 우선순위 규칙 충족"

    return "반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함"


def build_priority_reason(row):
    """표와 LLM 입력에 넣을 근거 문장을 만든다."""
    affected = int(row["affected_review_count"])
    neg_mixed = int(row["negative_mixed_review_count"])
    steam_negative = int(row["steam_negative_review_count"])
    recent_negative = int(row["recent_30d_negative_mixed_review_count"])
    early_negative = int(row["early_playtime_negative_mixed_review_count"])
    high = int(row["high_urgency_review_count"])

    parts = [
        f"영향 리뷰 {affected}개",
        f"부정·혼합 {neg_mixed}개",
        f"Steam 비추천 맥락 {steam_negative}개",
    ]

    if recent_negative > 0:
        parts.append(f"최근 30일 부정·혼합 {recent_negative}개")

    if early_negative > 0:
        parts.append(f"초반 플레이타임 부정·혼합 {early_negative}개")

    # High urgency는 우선순위 계산값이 아니라 보조 참고 지표로만 적는다.
    if high > 0:
        parts.append(f"High urgency 후보 {high}개(보조 참고)")

    if "priority_candidate" in row.index:
        parts.append(f"절대 기준 후보 {row['priority_candidate']}")

    parts.append(f"규칙 근거: {row['priority_rule_detail']}")
    return ", ".join(parts)


In [197]:
# ============================================================
# 이슈 단위 요약 생성
# ============================================================

def summarize_issue_group(g):
    affected = g["recommendationid"].nunique()

    positive_ids = g.loc[g["issue_positive_flag"], "recommendationid"].nunique()
    negative_ids = g.loc[g["issue_negative_flag"], "recommendationid"].nunique()
    mixed_ids = g.loc[g["issue_mixed_flag"], "recommendationid"].nunique()
    negative_mixed_ids = g.loc[g["issue_negative_or_mixed_flag"], "recommendationid"].nunique()

    high_ids = g.loc[g["high_urgency_flag"], "recommendationid"].nunique()
    high_negative_mixed_ids = g.loc[
        g["high_urgency_flag"] & g["issue_negative_or_mixed_flag"],
        "recommendationid"
    ].nunique()

    steam_negative_ids = g.loc[g["steam_negative_flag"], "recommendationid"].nunique()

    recent_30_ids = g.loc[g["recent_30d_flag"], "recommendationid"].nunique()
    recent_30_neg_mixed_ids = g.loc[
        g["recent_30d_flag"] & g["issue_negative_or_mixed_flag"],
        "recommendationid"
    ].nunique()

    early_neg_mixed_ids = g.loc[
        g["early_playtime_flag"] & g["issue_negative_or_mixed_flag"],
        "recommendationid"
    ].nunique()

    return pd.Series({
        "appid": g["appid"].iloc[0],
        "game_name": g["game_name"].iloc[0],
        "issue_name_kor": g["issue_name_kor"].iloc[0],
        "affected_review_count": affected,
        "positive_review_count": positive_ids,
        "negative_review_count": negative_ids,
        "mixed_review_count": mixed_ids,
        "negative_mixed_review_count": negative_mixed_ids,
        "steam_negative_review_count": steam_negative_ids,
        "high_urgency_review_count": high_ids,
        "high_urgency_negative_mixed_review_count": high_negative_mixed_ids,
        "recent_30d_review_count": recent_30_ids,
        "recent_30d_negative_mixed_review_count": recent_30_neg_mixed_ids,
        "early_playtime_negative_mixed_review_count": early_neg_mixed_ids,
        "avg_playtime_at_review_hours": round(g["playtime_at_review_hours"].mean(), 2),
        "median_playtime_at_review_hours": round(g["playtime_at_review_hours"].median(), 2),
    })


issue_summary = (
    issue_review_base
    .groupby("llm_issue_category", dropna=False)
    .apply(summarize_issue_group, include_groups=False)
    .reset_index()
)

issue_summary["high_urgency_rate"] = issue_summary.apply(
    lambda row: safe_rate(row["high_urgency_review_count"], row["affected_review_count"]),
    axis=1,
)
issue_summary["negative_mixed_rate"] = issue_summary.apply(
    lambda row: safe_rate(row["negative_mixed_review_count"], row["affected_review_count"]),
    axis=1,
)
issue_summary["steam_negative_rate"] = issue_summary.apply(
    lambda row: safe_rate(row["steam_negative_review_count"], row["affected_review_count"]),
    axis=1,
)
issue_summary["recent_30d_negative_mixed_rate"] = issue_summary.apply(
    lambda row: safe_rate(row["recent_30d_negative_mixed_review_count"], row["recent_30d_review_count"]),
    axis=1,
)

# ------------------------------------------------------------
# 대응 구분과 우선 검토 수준 계산
# ------------------------------------------------------------
# 1. action_group_hint: 이슈 성격 기반 대응 구분
# 2. priority_candidate: 절대 기준으로 만든 우선 검토 후보
# 3. rule_priority_hint: 후보값을 게임 내 상대 우선순위로 보정한 최종 상/중/하
issue_summary["action_group_hint"] = issue_summary.apply(get_action_group_hint, axis=1)
issue_summary["priority_candidate"] = issue_summary.apply(get_rule_priority_hint, axis=1)
issue_summary = apply_relative_priority(issue_summary)

issue_summary["priority_rule_detail"] = issue_summary.apply(get_priority_rule_detail, axis=1)
issue_summary["priority_reason"] = issue_summary.apply(build_priority_reason, axis=1)
issue_summary["patch_ops_note"] = issue_summary["llm_issue_category"].map(PATCH_OPS_NOTE_MAP).fillna("세부 리뷰 확인 후 대응 방향을 검토한다.")

# 보고서에서 보기 좋은 정렬
priority_order = {"상": 0, "중": 1, "하": 2}
action_order = {"즉시 확인": 0, "단기 개선": 1, "운영 커뮤니케이션 개선": 2, "장기 검토": 3, "검토 필요": 4, "강점 유지": 5}

issue_summary["priority_order"] = issue_summary["rule_priority_hint"].map(priority_order).fillna(9)
issue_summary["action_order"] = issue_summary["action_group_hint"].map(action_order).fillna(9)

issue_summary = issue_summary.sort_values(
    [
        "priority_order",
        "action_order",
        "negative_mixed_review_count",
        "steam_negative_review_count",
        "recent_30d_negative_mixed_review_count",
        "affected_review_count",
    ],
    ascending=[True, True, False, False, False, False],
).drop(columns=["priority_order", "action_order"])

print("이슈 단위 요약:", issue_summary.shape)
print("최종 우선 검토 수준 분포")
display(issue_summary["rule_priority_hint"].value_counts().rename_axis("rule_priority_hint").reset_index(name="issue_count"))
print("절대 기준 후보 분포")
display(issue_summary["priority_candidate"].value_counts().rename_axis("priority_candidate").reset_index(name="issue_count"))
display(issue_summary.head(20))


이슈 단위 요약: (17, 28)
최종 우선 검토 수준 분포


,rule_priority_hint,issue_count
0,하,10
1,중,6
2,상,1


절대 기준 후보 분포


,priority_candidate,issue_count
0,하,10
1,중,6
2,상,1


,llm_issue_category,appid,game_name,issue_name_kor,affected_review_count,positive_review_count,negative_review_count,mixed_review_count,negative_mixed_review_count,steam_negative_review_count,high_urgency_review_count,high_urgency_negative_mixed_review_count,recent_30d_review_count,recent_30d_negative_mixed_review_count,early_playtime_negative_mixed_review_count,avg_playtime_at_review_hours,median_playtime_at_review_hours,high_urgency_rate,negative_mixed_rate,steam_negative_rate,recent_30d_negative_mixed_rate,action_group_hint,priority_candidate,rule_priority_hint,priority_selection_note,priority_rule_detail,priority_reason,patch_ops_note
7,gameplay_loop,2990640,Endoparasitic 2,게임플레이 루프,87,20,55,4,59,35,22,21,0,0,21,7.87,6.45,0.2529,0.6782,0.4023,0.0,단기 개선,상,상,절대 기준 '상' 후보 중 이번 게임 내 상대 우선순위 상위 5개로 유지,절대 기준 상 후보 + 게임 내 상대 우선순위 상위 이슈,"영향 리뷰 87개, 부정·혼합 59개, Steam 비추천 맥락 35개, 초반 플레이타임 부정·혼합 21개, High urgency 후보 22개(보조 참고), 절대 기준 후보 상, 규칙 근거: 절대 기준 상 후...","반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다."
1,bug,2990640,Endoparasitic 2,버그,16,1,14,1,15,5,6,6,0,0,2,8.90,8.53,0.3750,0.9375,0.3125,0.0,즉시 확인,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,"영향 리뷰 16개, 부정·혼합 15개, Steam 비추천 맥락 5개, 초반 플레이타임 부정·혼합 2개, High urgency 후보 6개(보조 참고), 절대 기준 후보 중, 규칙 근거: 부정·혼합 맥락과 St...","반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다."
16,ui_ux,2990640,Endoparasitic 2,UI/UX,34,2,30,1,31,13,11,11,0,0,15,7.10,6.35,0.3235,0.9118,0.3824,0.0,단기 개선,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,"영향 리뷰 34개, 부정·혼합 31개, Steam 비추천 맥락 13개, 초반 플레이타임 부정·혼합 15개, High urgency 후보 11개(보조 참고), 절대 기준 후보 중, 규칙 근거: 부정·혼합 맥락과...","메뉴, 인벤토리, 퀘스트 안내, 조작 안내 등 편의성 문제를 개선한다."
0,balance,2990640,Endoparasitic 2,밸런스,32,1,26,2,28,9,8,8,0,0,5,9.18,8.24,0.2500,0.8750,0.2812,0.0,단기 개선,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,"영향 리뷰 32개, 부정·혼합 28개, Steam 비추천 맥락 9개, 초반 플레이타임 부정·혼합 5개, High urgency 후보 8개(보조 참고), 절대 기준 후보 중, 규칙 근거: 부정·혼합 맥락과 St...","전투, 성장, 보상, 적 난이도의 불균형 지점을 조정한다."
6,difficulty,2990640,Endoparasitic 2,난이도,26,3,19,0,19,9,5,5,0,0,2,10.09,10.15,0.1923,0.7308,0.3462,0.0,단기 개선,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,"영향 리뷰 26개, 부정·혼합 19개, Steam 비추천 맥락 9개, 초반 플레이타임 부정·혼합 2개, High urgency 후보 5개(보조 참고), 절대 기준 후보 중, 규칙 근거: 부정·혼합 맥락과 St...",초반 진입 장벽과 후반 난이도 피로를 구분해 난이도 옵션 또는 안내를 보강한다.
2,content_volume,2990640,Endoparasitic 2,콘텐츠 분량,33,1,18,1,19,12,4,3,1,1,4,8.96,8.72,0.1212,0.5758,0.3636,1.0,장기 검토,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,"영향 리뷰 33개, 부정·혼합 19개, Steam 비추천 맥락 12개, 최근 30일 부정·혼합 1개, 초반 플레이타임 부정·혼합 4개, High urgency 후보 4개(보조 참고), 절대 기준 후보 중, 규...",콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.
15,story,2990640,Endoparasitic 2,스토리,32,6,19,0,19,11,7,7,1,1,1,11.02,9.38,0.2188,0.5938,0.3438,1.0,장기 검토,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,"영향 리뷰 32개, 부정·혼합 19개, Steam 비추천 맥락 11개, 최근 30일 부정·혼합 1개, 초반 플레이타임 부정·혼합 1개, High urgency 후보 7개(보조 참고), 절대 기준 후보 중, 규...","서사 전달, 퀘스트 흐름, 엔딩/분기 만족도를 장기 개선 후보로 검토한다."
14,save_progression,2990640,Endoparasitic 2,저장/진행,6,0,5,0,5,3,3,3,0,0,3,6.08,4.31,0.5000,0.8333,0.5000,0.0,즉시 확인,하,하,절대 기준상 하 또는 개선 우선순위 근거가 상대적으로 약함,반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함,"영향 리뷰 6개, 부정·혼합 5개, Steam 비추천 맥락 3개, 초반 플레이타임 부정·혼합 3개, High urgency 후보 3개(보조 참고), 절대 기준 후보 하, 규칙 근거: 반복성 또는 Steam 비...","저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다."
3,control,2990640,Endoparasitic 2,조작감,8,2,5,0,5,4,4,4,0,0,3,8.90,5.68,0.5000,0.6250,0.5000,0.0,단기 개선,하,하,절대 기준상 하 또는 개선 우선순위 근거가 상대적으로 약함,반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함,"영향 리뷰 8개, 부정·혼합 5개, Steam 비추천 맥락 4개, 초반 플레이타임 부정·혼합 3개, High urgency 후보 4개(보조 참고), 절대 기준 후보 하, 규칙 근거: 반복성 또는 Steam 비...","이동, 전투, 상호작용 조작의 반응성과 키 설정 편의성을 점검한다."
8,graphics_audio,2990640,Endoparasitic 2,그래픽/사운드,15,4,8,1,9,7,6,5,0,0,2,8.10,7.75,0.4000,0.6000,0.4667,0.0,장기 검토,하,하,절대 기준상 하 또는 개선 우선순위 근거가 상대적으로 약함,반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함,"영향 리뷰 15개, 부정·혼합 9개, Steam 비추천 맥락 7개, 초반 플레이타임 부정·혼합 2개, High urgency 후보 6개(보조 참고), 절대 기준 후보 하, 규칙 근거: 반복성 또는 Steam ...",그래픽/사운드가 몰입을 방해하는지와 강점으로 작동하는지를 함께 확인한다.


# 7. 플레이타임/최근성 기준 이슈 요약

패치·운영 전략에서 다음 질문에 답하기 위한 보조 집계다.

- 초반 플레이에서 불만이 많은 이슈는 무엇인가?
- 최근 30일에도 반복되는 이슈는 무엇인가?

이 결과는 `postlaunch_issue_summary.csv`와 `postlaunch_patch_ops_evidence_base.csv`에 필요한 값으로 반영하고, 별도 CSV로는 저장하지 않는다.

In [198]:
# ============================================================
# 플레이타임 구간별 이슈 요약
# ============================================================

playtime_order = ["0-1h", "1-5h", "5-20h", "20-50h", "50h+", "unknown"]

playtime_issue_summary = (
    issue_review_base
    .groupby(["playtime_stage", "llm_issue_category", "issue_name_kor"], dropna=False)
    .agg(
        affected_review_count=("recommendationid", "nunique"),
        negative_mixed_review_count=("issue_negative_or_mixed_flag", "sum"),
        high_urgency_review_count=("high_urgency_flag", "sum"),
        steam_negative_review_count=("steam_negative_flag", "sum"),
    )
    .reset_index()
)

playtime_issue_summary["playtime_stage"] = pd.Categorical(
    playtime_issue_summary["playtime_stage"],
    categories=playtime_order,
    ordered=True,
)
playtime_issue_summary = playtime_issue_summary.sort_values(
    ["playtime_stage", "negative_mixed_review_count", "high_urgency_review_count"],
    ascending=[True, False, False],
)

print("플레이타임 구간별 이슈 요약:", playtime_issue_summary.shape)
display(playtime_issue_summary.head(20))


플레이타임 구간별 이슈 요약: (47, 7)


,playtime_stage,llm_issue_category,issue_name_kor,affected_review_count,negative_mixed_review_count,high_urgency_review_count,steam_negative_review_count
3,0-1h,gameplay_loop,게임플레이 루프,5,5,1,5
4,0-1h,other,기타,3,2,0,1
0,0-1h,content_volume,콘텐츠 분량,2,1,0,1
2,0-1h,difficulty,난이도,1,1,0,1
1,0-1h,control,조작감,1,0,0,0
5,0-1h,positive_praise,긍정 칭찬,8,0,0,0
11,1-5h,gameplay_loop,게임플레이 루프,22,16,9,14
19,1-5h,ui_ux,UI/UX,15,15,7,10
6,1-5h,balance,밸런스,6,5,3,3
9,1-5h,control,조작감,3,3,3,3


In [199]:
# ============================================================
# 최근성 구간별 이슈 요약
# ============================================================

recency_order = ["last_30d", "31-60d", "61-90d", "older_90d", "unknown"]

recency_issue_summary = (
    issue_review_base
    .groupby(["review_recency_group", "llm_issue_category", "issue_name_kor"], dropna=False)
    .agg(
        affected_review_count=("recommendationid", "nunique"),
        negative_mixed_review_count=("issue_negative_or_mixed_flag", "sum"),
        high_urgency_review_count=("high_urgency_flag", "sum"),
        steam_negative_review_count=("steam_negative_flag", "sum"),
    )
    .reset_index()
)

recency_issue_summary["review_recency_group"] = pd.Categorical(
    recency_issue_summary["review_recency_group"],
    categories=recency_order,
    ordered=True,
)
recency_issue_summary = recency_issue_summary.sort_values(
    ["review_recency_group", "negative_mixed_review_count", "high_urgency_review_count"],
    ascending=[True, False, False],
)

print("최근성 구간별 이슈 요약:", recency_issue_summary.shape)
display(recency_issue_summary.head(20))


최근성 구간별 이슈 요약: (25, 7)


,review_recency_group,llm_issue_category,issue_name_kor,affected_review_count,negative_mixed_review_count,high_urgency_review_count,steam_negative_review_count
4,last_30d,content_volume,콘텐츠 분량,1,1,0,1
5,last_30d,other,기타,1,1,0,1
7,last_30d,story,스토리,1,1,0,0
6,last_30d,positive_praise,긍정 칭찬,1,0,0,0
0,61-90d,difficulty,난이도,2,2,1,1
1,61-90d,gameplay_loop,게임플레이 루프,1,1,1,1
2,61-90d,other,기타,1,1,1,1
3,61-90d,positive_praise,긍정 칭찬,1,0,0,0
15,older_90d,gameplay_loop,게임플레이 루프,86,58,21,34
24,older_90d,ui_ux,UI/UX,34,31,11,13


# 8. 04-2 LLM 입력용 근거 데이터 생성

04-2에서는 이 데이터를 LLM에게 제공해 **패치·운영 전략 초안**을 만들 예정이다.

핵심 원칙은 다음과 같다.

- LLM에게 개별 리뷰만 주고 최종 우선순위를 판단하게 하지 않는다.
- 04-1에서 만든 반복 이슈 근거, 부정·혼합 분포, Steam 비추천 맥락, 최근성, 플레이타임 근거를 함께 제공한다.
- `High urgency`는 04번 LLM 리뷰 분류에서 나온 보조 참고 지표로만 전달한다.
- 04-2의 LLM은 `action_group_hint`와 `rule_priority_hint`를 새로 판단하지 않고, 개발자가 읽기 쉬운 문장으로 정리하는 역할만 한다.


In [200]:
# ============================================================
# 04-2 LLM 입력용 근거 문장 생성 함수
# ============================================================

def clean_evidence_text(x, max_len=180):
    text = normalize_text_value(x)
    text = re.sub(r"\s+", " ", text)
    if len(text) > max_len:
        return text[:max_len].rstrip() + "..."
    return text


def collect_examples(issue_category, max_examples=5):
    """이슈별 대표 리뷰 요약/근거/개선 제안을 가져온다."""
    g = issue_review_base[issue_review_base["llm_issue_category"] == issue_category].copy()

    if len(g) == 0:
        return ""

    # 부정/혼합 + 최근 리뷰 + Steam 비추천 리뷰를 우선적으로 보여준다.
    # High urgency는 보조 참고 지표이므로 대표 리뷰 정렬에서 가장 앞 기준으로 쓰지 않는다.
    g["sort_negative"] = g["issue_negative_or_mixed_flag"].astype(int)
    g["sort_steam_negative"] = g["steam_negative_flag"].astype(int)
    g["sort_recent"] = g["recent_30d_flag"].astype(int)
    g["sort_votes"] = pd.to_numeric(g.get("votes_up", 0), errors="coerce").fillna(0)

    g = g.sort_values(
        ["sort_negative", "sort_steam_negative", "sort_recent", "sort_votes"],
        ascending=[False, False, False, False],
    )

    examples = []
    used = set()

    for _, row in g.iterrows():
        rec_id = row["recommendationid"]
        if rec_id in used:
            continue
        used.add(rec_id)

        evidence = clean_evidence_text(row.get("llm_issue_evidence", ""))
        summary = clean_evidence_text(row.get("llm_review_summary", ""))
        suggested = clean_evidence_text(row.get("llm_suggested_action", ""))

        example = (
            f"- steam_label={row.get('steam_label_text', '')}, "
            f"issue_sentiment={row.get('llm_issue_sentiment', '')}, "
            f"urgency_candidate={row.get('llm_urgency_candidate', '')}, "
            f"playtime={row.get('playtime_stage', '')}, "
            f"recency={row.get('review_recency_group', '')} | "
            f"근거: {evidence} | 요약: {summary} | LLM 개선 제안 후보: {suggested}"
        )
        examples.append(example)

        if len(examples) >= max_examples:
            break

    return "\n".join(examples)


def build_llm_evidence_text(row):
    """04-2 프롬프트에 바로 넣기 쉬운 이슈별 근거 블록을 만든다."""
    examples = collect_examples(row["llm_issue_category"], max_examples=5)

    return f"""
[ISSUE]
issue_category: {row['llm_issue_category']}
issue_name_kor: {row['issue_name_kor']}
action_group_hint: {row['action_group_hint']}
priority_candidate: {row.get('priority_candidate', '')}  # 절대 기준 후보
rule_priority_hint: {row['rule_priority_hint']}  # 상대 보정 후 최종 우선 검토 수준
priority_selection_note: {row.get('priority_selection_note', '')}
priority_rule_detail: {row['priority_rule_detail']}
affected_review_count: {int(row['affected_review_count'])}
negative_mixed_review_count: {int(row['negative_mixed_review_count'])}
steam_negative_review_count: {int(row['steam_negative_review_count'])}
recent_30d_negative_mixed_review_count: {int(row['recent_30d_negative_mixed_review_count'])}
early_playtime_negative_mixed_review_count: {int(row['early_playtime_negative_mixed_review_count'])}
high_urgency_review_count: {int(row['high_urgency_review_count'])}  # 보조 참고 지표
high_urgency_rate: {row['high_urgency_rate']}  # 보조 참고 지표
priority_reason: {row['priority_reason']}
patch_ops_note: {row['patch_ops_note']}
대표 근거:
{examples}
[/ISSUE]
""".strip()


In [201]:
# ============================================================
# 04-2 LLM 입력용 근거 테이블 생성
# ============================================================

patch_ops_evidence_base = issue_summary.copy()
patch_ops_evidence_base["llm_evidence_text"] = patch_ops_evidence_base.apply(build_llm_evidence_text, axis=1)

# 04-2에서 너무 많은 이슈를 모두 넣지 않도록 기본 정렬 상태로 저장한다.
# 필요하면 04-2에서 상/중 우선순위만 필터링해서 사용할 수 있다.

print("패치·운영 전략 생성용 근거 테이블:", patch_ops_evidence_base.shape)
display(patch_ops_evidence_base[[
    "llm_issue_category", "issue_name_kor", "action_group_hint",
    "priority_candidate", "rule_priority_hint", "priority_selection_note",
    "priority_rule_detail", "affected_review_count", "negative_mixed_review_count",
    "steam_negative_review_count", "high_urgency_review_count",
    "recent_30d_negative_mixed_review_count", "early_playtime_negative_mixed_review_count",
    "priority_reason", "patch_ops_note",
]].head(20))


패치·운영 전략 생성용 근거 테이블: (17, 29)


,llm_issue_category,issue_name_kor,action_group_hint,priority_candidate,rule_priority_hint,priority_selection_note,priority_rule_detail,affected_review_count,negative_mixed_review_count,steam_negative_review_count,high_urgency_review_count,recent_30d_negative_mixed_review_count,early_playtime_negative_mixed_review_count,priority_reason,patch_ops_note
7,gameplay_loop,게임플레이 루프,단기 개선,상,상,절대 기준 '상' 후보 중 이번 게임 내 상대 우선순위 상위 5개로 유지,절대 기준 상 후보 + 게임 내 상대 우선순위 상위 이슈,87,59,35,22,0,21,"영향 리뷰 87개, 부정·혼합 59개, Steam 비추천 맥락 35개, 초반 플레이타임 부정·혼합 21개, High urgency 후보 22개(보조 참고), 절대 기준 후보 상, 규칙 근거: 절대 기준 상 후...","반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다."
1,bug,버그,즉시 확인,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,16,15,5,6,0,2,"영향 리뷰 16개, 부정·혼합 15개, Steam 비추천 맥락 5개, 초반 플레이타임 부정·혼합 2개, High urgency 후보 6개(보조 참고), 절대 기준 후보 중, 규칙 근거: 부정·혼합 맥락과 St...","반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다."
16,ui_ux,UI/UX,단기 개선,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,34,31,13,11,0,15,"영향 리뷰 34개, 부정·혼합 31개, Steam 비추천 맥락 13개, 초반 플레이타임 부정·혼합 15개, High urgency 후보 11개(보조 참고), 절대 기준 후보 중, 규칙 근거: 부정·혼합 맥락과...","메뉴, 인벤토리, 퀘스트 안내, 조작 안내 등 편의성 문제를 개선한다."
0,balance,밸런스,단기 개선,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,32,28,9,8,0,5,"영향 리뷰 32개, 부정·혼합 28개, Steam 비추천 맥락 9개, 초반 플레이타임 부정·혼합 5개, High urgency 후보 8개(보조 참고), 절대 기준 후보 중, 규칙 근거: 부정·혼합 맥락과 St...","전투, 성장, 보상, 적 난이도의 불균형 지점을 조정한다."
6,difficulty,난이도,단기 개선,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,26,19,9,5,0,2,"영향 리뷰 26개, 부정·혼합 19개, Steam 비추천 맥락 9개, 초반 플레이타임 부정·혼합 2개, High urgency 후보 5개(보조 참고), 절대 기준 후보 중, 규칙 근거: 부정·혼합 맥락과 St...",초반 진입 장벽과 후반 난이도 피로를 구분해 난이도 옵션 또는 안내를 보강한다.
2,content_volume,콘텐츠 분량,장기 검토,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,33,19,12,4,1,4,"영향 리뷰 33개, 부정·혼합 19개, Steam 비추천 맥락 12개, 최근 30일 부정·혼합 1개, 초반 플레이타임 부정·혼합 4개, High urgency 후보 4개(보조 참고), 절대 기준 후보 중, 규...",콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.
15,story,스토리,장기 검토,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,32,19,11,7,1,1,"영향 리뷰 32개, 부정·혼합 19개, Steam 비추천 맥락 11개, 최근 30일 부정·혼합 1개, 초반 플레이타임 부정·혼합 1개, High urgency 후보 7개(보조 참고), 절대 기준 후보 중, 규...","서사 전달, 퀘스트 흐름, 엔딩/분기 만족도를 장기 개선 후보로 검토한다."
14,save_progression,저장/진행,즉시 확인,하,하,절대 기준상 하 또는 개선 우선순위 근거가 상대적으로 약함,반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함,6,5,3,3,0,3,"영향 리뷰 6개, 부정·혼합 5개, Steam 비추천 맥락 3개, 초반 플레이타임 부정·혼합 3개, High urgency 후보 3개(보조 참고), 절대 기준 후보 하, 규칙 근거: 반복성 또는 Steam 비...","저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다."
3,control,조작감,단기 개선,하,하,절대 기준상 하 또는 개선 우선순위 근거가 상대적으로 약함,반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함,8,5,4,4,0,3,"영향 리뷰 8개, 부정·혼합 5개, Steam 비추천 맥락 4개, 초반 플레이타임 부정·혼합 3개, High urgency 후보 4개(보조 참고), 절대 기준 후보 하, 규칙 근거: 반복성 또는 Steam 비...","이동, 전투, 상호작용 조작의 반응성과 키 설정 편의성을 점검한다."
8,graphics_audio,그래픽/사운드,장기 검토,하,하,절대 기준상 하 또는 개선 우선순위 근거가 상대적으로 약함,반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함,15,9,7,6,0,2,"영향 리뷰 15개, 부정·혼합 9개, Steam 비추천 맥락 7개, 초반 플레이타임 부정·혼합 2개, High urgency 후보 6개(보조 참고), 절대 기준 후보 하, 규칙 근거: 반복성 또는 Steam ...",그래픽/사운드가 몰입을 방해하는지와 강점으로 작동하는지를 함께 확인한다.


# 9. Tableau/보고서용 원천 데이터 생성

이슈 태그 단위 원천 데이터에 이슈 요약값을 붙여, 대시보드에서 필터/집계를 하기 쉽게 만든다.


In [202]:
# ============================================================
# Tableau/보고서용 원천 데이터 생성
# ============================================================

summary_cols_for_merge = [
    "llm_issue_category", "action_group_hint", "priority_candidate", "rule_priority_hint",
    "priority_selection_note", "priority_rule_detail", "priority_reason",
    "affected_review_count", "negative_mixed_review_count", "steam_negative_review_count",
    "high_urgency_review_count", "high_urgency_rate",
    "negative_mixed_rate", "steam_negative_rate",
    "recent_30d_negative_mixed_review_count",
    "early_playtime_negative_mixed_review_count",
    "patch_ops_note",
]

summary_cols_for_merge = [col for col in summary_cols_for_merge if col in issue_summary.columns]

tableau_source = issue_review_base.merge(
    issue_summary[summary_cols_for_merge],
    on="llm_issue_category",
    how="left",
    suffixes=("", "_issue_summary"),
)

# Tableau에서 쓰기 좋은 컬럼만 선택
keep_cols = [
    "appid", "game_name", "recommendationid",
    "review_datetime", "release_date", "days_from_release", "release_period", "review_recency_group",
    "steam_label_text", "llm_sentiment", "steam_llm_sentiment_relation",
    "playtime_at_review_hours", "playtime_stage", "early_playtime_flag",
    "votes_up", "weighted_vote_score",
    "llm_urgency_candidate", "high_urgency_flag",
    "llm_issue_category", "issue_name_kor", "llm_issue_sentiment",
    "issue_positive_flag", "issue_negative_or_mixed_flag",
    "action_group_hint", "priority_candidate", "rule_priority_hint",
    "priority_selection_note", "priority_rule_detail", "priority_reason",
    "affected_review_count", "negative_mixed_review_count", "steam_negative_review_count",
    "high_urgency_review_count", "high_urgency_rate", "negative_mixed_rate", "steam_negative_rate",
    "recent_30d_negative_mixed_review_count", "early_playtime_negative_mixed_review_count",
    "patch_ops_note", "llm_issue_evidence",
]
keep_cols = [col for col in keep_cols if col in tableau_source.columns]
tableau_source = tableau_source[keep_cols].copy()

print("Tableau/보고서 원천 데이터:", tableau_source.shape)
display(tableau_source.head())


Tableau/보고서 원천 데이터: (477, 40)


,appid,game_name,recommendationid,review_datetime,release_date,days_from_release,release_period,review_recency_group,steam_label_text,llm_sentiment,steam_llm_sentiment_relation,playtime_at_review_hours,playtime_stage,early_playtime_flag,votes_up,weighted_vote_score,llm_urgency_candidate,high_urgency_flag,llm_issue_category,issue_name_kor,llm_issue_sentiment,issue_positive_flag,issue_negative_or_mixed_flag,action_group_hint,priority_candidate,rule_priority_hint,priority_selection_note,priority_rule_detail,priority_reason,affected_review_count,negative_mixed_review_count,steam_negative_review_count,high_urgency_review_count,high_urgency_rate,negative_mixed_rate,steam_negative_rate,recent_30d_negative_mixed_review_count,early_playtime_negative_mixed_review_count,patch_ops_note,llm_issue_evidence
0,2990640,Endoparasitic 2,176177053,2024-10-01 20:00:52,2024-10-01,0,D0-D30,older_90d,positive,positive,exact_match,0.266667,0-1h,True,6,0.510040,low,False,positive_praise,긍정 칭찬,positive,True,False,강점 유지,하,하,강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리,강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리,"영향 리뷰 164개, 부정·혼합 0개, Steam 비추천 맥락 4개, High urgency 후보 6개(보조 참고), 절대 기준 후보 하, 규칙 근거: 강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리",164,0,4,6,0.0366,0.0000,0.0244,0,0,"긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.",한 손으로 플레이 가능한 조작감과 훌륭한 속편이라는 점을 칭찬함
1,2990640,Endoparasitic 2,176177300,2024-10-01 20:05:29,2024-10-01,0,D0-D30,older_90d,positive,positive,exact_match,0.383333,0-1h,True,1,0.490937,low,False,positive_praise,긍정 칭찬,positive,True,False,강점 유지,하,하,강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리,강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리,"영향 리뷰 164개, 부정·혼합 0개, Steam 비추천 맥락 4개, High urgency 후보 6개(보조 참고), 절대 기준 후보 하, 규칙 근거: 강점 유지 항목이므로 개선 우선순위 산정 대상에서 분리",164,0,4,6,0.0366,0.0000,0.0244,0,0,"긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.",게임이 매우 훌륭하다고 언급함
2,2990640,Endoparasitic 2,176179490,2024-10-01 20:46:59,2024-10-01,0,D0-D30,older_90d,positive,positive,exact_match,10.433333,5-20h,False,34,0.749335,medium,False,content_volume,콘텐츠 분량,neutral,False,False,장기 검토,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,"영향 리뷰 33개, 부정·혼합 19개, Steam 비추천 맥락 12개, 최근 30일 부정·혼합 1개, 초반 플레이타임 부정·혼합 4개, High urgency 후보 4개(보조 참고), 절대 기준 후보 중, 규...",33,19,12,4,0.1212,0.5758,0.3636,1,4,콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.,적 유형과 무기 종류가 각각 3개뿐이라는 점을 지적함
3,2990640,Endoparasitic 2,176179490,2024-10-01 20:46:59,2024-10-01,0,D0-D30,older_90d,positive,positive,exact_match,10.433333,5-20h,False,34,0.749335,medium,False,gameplay_loop,게임플레이 루프,positive,True,False,단기 개선,상,상,절대 기준 '상' 후보 중 이번 게임 내 상대 우선순위 상위 5개로 유지,절대 기준 상 후보 + 게임 내 상대 우선순위 상위 이슈,"영향 리뷰 87개, 부정·혼합 59개, Steam 비추천 맥락 35개, 초반 플레이타임 부정·혼합 21개, High urgency 후보 22개(보조 참고), 절대 기준 후보 상, 규칙 근거: 절대 기준 상 후...",87,59,35,22,0.2529,0.6782,0.4023,0,21,"반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다.",크래프팅 시스템이 단순하지만 잘 작동하고 재미있음
4,2990640,Endoparasitic 2,176179490,2024-10-01 20:46:59,2024-10-01,0,D0-D30,older_90d,positive,positive,exact_match,10.433333,5-20h,False,34,0.749335,medium,False,story,스토리,positive,True,False,장기 검토,중,중,절대 기준상 '중' 후보로 분류,부정·혼합 맥락과 Steam 비추천 맥락이 일정 수준 확인,"영향 리뷰 32개, 부정·혼합 19개, Steam 비추천 맥락 11개, 최근 30일 부정·혼합 1개, 초반 플레이타임 부정·혼합 1개, High urgency 후보 7개(보조 참고), 절대 기준 후보 중, 규...",32,19,11,7,0.2188,0.5938,0.3438,1,1,"서사 전달, 퀘스트 흐름, 엔딩/분기 만족도를 장기 개선 후보로 검토한다.",스토리가 전작보다 깊이 있고 재미있음


# 10. CSV 저장

In [203]:
# ============================================================
# CSV 저장
# ============================================================
# 04-1 산출물은 04-2에서 실제로 사용할 최소 파일만 저장한다.

review_base.to_csv(POSTLAUNCH_REVIEW_BASE_PATH, index=False, encoding="utf-8-sig")
issue_summary.to_csv(POSTLAUNCH_ISSUE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
patch_ops_evidence_base.to_csv(POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH, index=False, encoding="utf-8-sig")
tableau_source.to_csv(TABLEAU_POSTLAUNCH_SOURCE_PATH, index=False, encoding="utf-8-sig")

saved_files = pd.DataFrame([
    {"파일명": POSTLAUNCH_REVIEW_BASE_PATH.name, "행 수": len(review_base), "역할": "리뷰 1개 단위 전처리 결과"},
    {"파일명": POSTLAUNCH_ISSUE_SUMMARY_PATH.name, "행 수": len(issue_summary), "역할": "이슈별 반복성/부정·혼합/Steam 비추천/High urgency 보조 지표 요약"},
    {"파일명": POSTLAUNCH_PATCH_OPS_EVIDENCE_BASE_PATH.name, "행 수": len(patch_ops_evidence_base), "역할": "04-2 LLM 패치·운영 전략 생성용 근거 데이터"},
    {"파일명": TABLEAU_POSTLAUNCH_SOURCE_PATH.name, "행 수": len(tableau_source), "역할": "Tableau/보고서용 원천 데이터"},
])

print("저장 완료")
print("저장 폴더:", POSTLAUNCH_PREPROCESS_DIR)
display(saved_files)

저장 완료
저장 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch\runs\endoparasitic_2\postlaunch_preprocess_data


,파일명,행 수,역할
0,postlaunch_review_base.csv,245,리뷰 1개 단위 전처리 결과
1,postlaunch_issue_summary.csv,17,이슈별 반복성/부정·혼합/Steam 비추천/High urgency 보조 지표 요약
2,postlaunch_patch_ops_evidence_base.csv,17,04-2 LLM 패치·운영 전략 생성용 근거 데이터
3,tableau_postlaunch_patch_ops_source.csv,477,Tableau/보고서용 원천 데이터


# 11. 우선순위 산정 로직 검증

In [204]:
# ============================================================
# 우선순위 산정 로직 검증
# ============================================================
# 목적:
# 튜터님 피드백이 코드에 실제로 반영되었는지 확인한다.
#
# 검증 방향:
# 1. LLM의 High urgency가 rule_priority_hint 계산에 직접 들어가지 않았는지 확인한다.
# 2. 최종 '상'은 반드시 rule 기반 priority_candidate == '상' 후보에서만 나오도록 확인한다.
# 3. '즉시 확인' 이슈는 기술/진행 차단 가능성이 있는 이슈로만 제한한다.
# 4. 리뷰 수가 많은 게임에서 '상'이 과도하게 많이 나오지 않도록 상대 보정이 적용되었는지 확인한다.
# 5. 04-2 입력용 근거 텍스트가 비어 있지 않은지 확인한다.

# ------------------------------------------------------------
# 0. 검증용 기준
# ------------------------------------------------------------
# get_rule_priority_hint 내부의 blocking 예외 기준과 맞춘다.
# 즉시 확인 이슈는 크래시/저장/진행/버그/성능처럼 플레이를 직접 막을 수 있으므로
# 일반 '상' 기준보다 부정·혼합 기준을 조금 낮게 둔다.
BLOCKING_HIGH_MIN_NEGATIVE_MIXED_REVIEWS = 15
BLOCKING_HIGH_MIN_STEAM_NEGATIVE_REVIEWS = 10

FORBIDDEN_PRIORITY_REFS = {
    "high_urgency_review_count",
    "high_urgency",
}

# ------------------------------------------------------------
# 1. 리뷰 단위 결과는 recommendationid가 중복되면 안 된다.
# ------------------------------------------------------------
assert review_base["recommendationid"].is_unique, "review_base에 recommendationid 중복이 있습니다."

# ------------------------------------------------------------
# 2. 이슈 요약의 affected_review_count는 전체 분석 리뷰 수보다 클 수 없다.
# ------------------------------------------------------------
total_review_count = review_base["recommendationid"].nunique()

affected_invalid = issue_summary[
    issue_summary["affected_review_count"] > total_review_count
]
assert len(affected_invalid) == 0, affected_invalid[[
    "llm_issue_category",
    "issue_name_kor",
    "affected_review_count",
]]

# ------------------------------------------------------------
# 3. 부정·혼합 리뷰 수는 영향 리뷰 수보다 클 수 없다.
# ------------------------------------------------------------
negative_count_invalid = issue_summary[
    issue_summary["negative_mixed_review_count"] > issue_summary["affected_review_count"]
]
assert len(negative_count_invalid) == 0, negative_count_invalid[[
    "llm_issue_category",
    "issue_name_kor",
    "affected_review_count",
    "negative_mixed_review_count",
]]

# ------------------------------------------------------------
# 4. High urgency 리뷰 수는 영향 리뷰 수보다 클 수 없다.
# ------------------------------------------------------------
high_urgency_count_invalid = issue_summary[
    issue_summary["high_urgency_review_count"] > issue_summary["affected_review_count"]
]
assert len(high_urgency_count_invalid) == 0, high_urgency_count_invalid[[
    "llm_issue_category",
    "issue_name_kor",
    "affected_review_count",
    "high_urgency_review_count",
]]

# ------------------------------------------------------------
# 5. get_rule_priority_hint 함수 내부에서 High urgency 계열 컬럼을 직접 참조하지 않는지 확인한다.
# ------------------------------------------------------------
priority_func_names = {str(x) for x in get_rule_priority_hint.__code__.co_names}
priority_func_consts = {
    str(x)
    for x in get_rule_priority_hint.__code__.co_consts
    if isinstance(x, str)
}
priority_func_refs = priority_func_names | priority_func_consts

priority_forbidden_refs = [
    ref
    for ref in priority_func_refs
    if any(forbidden in ref for forbidden in FORBIDDEN_PRIORITY_REFS)
]

assert len(priority_forbidden_refs) == 0, (
    "get_rule_priority_hint 계산 함수에서 High urgency 계열 참조가 발견되었습니다: "
    f"{priority_forbidden_refs}"
)

# ------------------------------------------------------------
# 6. apply_relative_priority 함수 내부에서도 High urgency 계열 컬럼을 직접 참조하지 않는지 확인한다.
# ------------------------------------------------------------
relative_func_names = {str(x) for x in apply_relative_priority.__code__.co_names}
relative_func_consts = {
    str(x)
    for x in apply_relative_priority.__code__.co_consts
    if isinstance(x, str)
}
relative_func_refs = relative_func_names | relative_func_consts

relative_forbidden_refs = [
    ref
    for ref in relative_func_refs
    if any(forbidden in ref for forbidden in FORBIDDEN_PRIORITY_REFS)
]

assert len(relative_forbidden_refs) == 0, (
    "apply_relative_priority 계산 함수에서 High urgency 계열 참조가 발견되었습니다: "
    f"{relative_forbidden_refs}"
)

# ------------------------------------------------------------
# 7. priority_candidate == '상'은 명확한 rule 근거가 있어야 한다.
# ------------------------------------------------------------
# 일반 '상' 기준 1:
# 부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 부정·혼합 반복
candidate_high_by_recent = (
    (issue_summary["negative_mixed_review_count"] >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS)
    & (issue_summary["steam_negative_review_count"] >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS)
    & (issue_summary["recent_30d_negative_mixed_review_count"] >= HIGH_PRIORITY_MIN_RECENT_NEGATIVE_MIXED_REVIEWS)
)

# 일반 '상' 기준 2:
# 영향 리뷰 규모가 크고, 부정·혼합 + Steam 비추천 근거가 함께 있는 경우
candidate_high_by_volume = (
    (issue_summary["affected_review_count"] >= HIGH_PRIORITY_MIN_AFFECTED_REVIEWS)
    & (issue_summary["negative_mixed_review_count"] >= HIGH_PRIORITY_MIN_NEGATIVE_MIXED_REVIEWS)
    & (issue_summary["steam_negative_review_count"] >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_REVIEWS)
)

# 즉시 확인 예외 기준:
# 기술/진행 차단 가능성이 있는 이슈는 일반 기준보다 낮은 부정·혼합 기준을 허용한다.
candidate_high_by_blocking_issue = (
    (issue_summary["action_group_hint"] == "즉시 확인")
    & (issue_summary["negative_mixed_review_count"] >= BLOCKING_HIGH_MIN_NEGATIVE_MIXED_REVIEWS)
    & (issue_summary["steam_negative_review_count"] >= BLOCKING_HIGH_MIN_STEAM_NEGATIVE_REVIEWS)
)

candidate_high_has_rule_basis = (
    candidate_high_by_recent
    | candidate_high_by_volume
    | candidate_high_by_blocking_issue
)

candidate_high_without_rule_basis = issue_summary[
    (issue_summary["priority_candidate"] == "상")
    & ~candidate_high_has_rule_basis
]

assert len(candidate_high_without_rule_basis) == 0, candidate_high_without_rule_basis[[
    "llm_issue_category",
    "issue_name_kor",
    "action_group_hint",
    "affected_review_count",
    "negative_mixed_review_count",
    "steam_negative_review_count",
    "recent_30d_negative_mixed_review_count",
    "priority_candidate",
]]

# ------------------------------------------------------------
# 8. 최종 '상'은 반드시 priority_candidate == '상' 후보에서만 나와야 한다.
# ------------------------------------------------------------
final_high_without_candidate = issue_summary[
    (issue_summary["rule_priority_hint"] == "상")
    & (issue_summary["priority_candidate"] != "상")
]

assert len(final_high_without_candidate) == 0, final_high_without_candidate[[
    "llm_issue_category",
    "issue_name_kor",
    "priority_candidate",
    "rule_priority_hint",
    "priority_selection_note",
]]

# ------------------------------------------------------------
# 9. High urgency만 많고 rule 근거가 없는데 최종 '상'으로 올라간 이슈가 없어야 한다.
# ------------------------------------------------------------
# 여기서는 부정·혼합 20개 미만을 무조건 금지하지 않는다.
# save_progression 같은 즉시 확인 이슈는 blocking 기준을 만족하면 '상'이 될 수 있기 때문이다.
high_urgency_only_high = issue_summary[
    (issue_summary["rule_priority_hint"] == "상")
    & (issue_summary["high_urgency_review_count"] >= 10)
    & ~candidate_high_has_rule_basis
]

assert len(high_urgency_only_high) == 0, high_urgency_only_high[[
    "llm_issue_category",
    "issue_name_kor",
    "action_group_hint",
    "high_urgency_review_count",
    "affected_review_count",
    "negative_mixed_review_count",
    "steam_negative_review_count",
    "recent_30d_negative_mixed_review_count",
    "priority_candidate",
    "rule_priority_hint",
]]

# ------------------------------------------------------------
# 10. 즉시 확인은 기술/진행 차단 가능성이 있는 이슈 그룹으로만 제한한다.
# ------------------------------------------------------------
immediate_invalid = issue_summary[
    (issue_summary["action_group_hint"] == "즉시 확인")
    & (~issue_summary["llm_issue_category"].isin(IMMEDIATE_ISSUES))
]

assert len(immediate_invalid) == 0, immediate_invalid[[
    "llm_issue_category",
    "issue_name_kor",
    "action_group_hint",
]]

# ------------------------------------------------------------
# 11. 강점 유지 항목은 개선 우선순위 '상'으로 들어가면 안 된다.
# ------------------------------------------------------------
strength_high = issue_summary[
    (issue_summary["action_group_hint"] == "강점 유지")
    & (issue_summary["rule_priority_hint"] == "상")
]

assert len(strength_high) == 0, strength_high[[
    "llm_issue_category",
    "issue_name_kor",
    "action_group_hint",
    "rule_priority_hint",
]]

# ------------------------------------------------------------
# 12. 기타 이슈는 원인이 명확하지 않으므로 최종 '상'으로 들어가면 안 된다.
# ------------------------------------------------------------
other_high = issue_summary[
    (issue_summary["llm_issue_category"] == "other")
    & (issue_summary["rule_priority_hint"] == "상")
]

assert len(other_high) == 0, other_high[[
    "llm_issue_category",
    "issue_name_kor",
    "rule_priority_hint",
    "priority_selection_note",
]]

# ------------------------------------------------------------
# 13. 최종 '상' 개수는 상대 보정 기준을 넘으면 안 된다.
# ------------------------------------------------------------
priority_target_count = int((
    issue_summary["action_group_hint"].ne("강점 유지")
    & ~issue_summary["llm_issue_category"].isin(STRENGTH_ISSUES)
).sum())

max_high_count = min(
    MAX_HIGH_ISSUES,
    max(1, int(np.ceil(priority_target_count * MAX_HIGH_RATE)))
)

actual_high_count = int((issue_summary["rule_priority_hint"] == "상").sum())

assert actual_high_count <= max_high_count, (
    f"최종 '상' 이슈 수가 제한 기준을 초과했습니다: {actual_high_count} > {max_high_count}"
)

# ------------------------------------------------------------
# 14. 절대 기준 상 후보였으나 최종 상이 아닌 항목은 중으로 조정되어야 한다.
# ------------------------------------------------------------
# 단, 강점 유지/기타 이슈는 별도 분리 대상이므로 예외로 둔다.
demoted_invalid = issue_summary[
    (issue_summary["priority_candidate"] == "상")
    & (issue_summary["rule_priority_hint"] == "하")
    & (issue_summary["action_group_hint"] != "강점 유지")
    & (~issue_summary["llm_issue_category"].isin(NO_HIGH_ISSUES))
]

assert len(demoted_invalid) == 0, demoted_invalid[[
    "llm_issue_category",
    "issue_name_kor",
    "priority_candidate",
    "rule_priority_hint",
    "priority_selection_note",
]]

# ------------------------------------------------------------
# 15. 04-2 입력용 근거 텍스트가 비어 있지 않은지 확인한다.
# ------------------------------------------------------------
evidence_empty = patch_ops_evidence_base[
    patch_ops_evidence_base["llm_evidence_text"].isna()
    | patch_ops_evidence_base["llm_evidence_text"].astype(str).str.strip().eq("")
]

assert len(evidence_empty) == 0, evidence_empty[[
    "llm_issue_category",
    "issue_name_kor",
    "llm_evidence_text",
]]

# ------------------------------------------------------------
# 검증 결과 출력
# ------------------------------------------------------------
print("검증 통과")
print("리뷰 수:", review_base["recommendationid"].nunique())
print("리뷰-이슈 수:", len(issue_review_base))
print("이슈 종류 수:", issue_summary["llm_issue_category"].nunique())

print("절대 기준 후보 분포:")
display(
    issue_summary["priority_candidate"]
    .value_counts()
    .rename_axis("priority_candidate")
    .reset_index(name="issue_count")
)

print("최종 우선 검토 수준 분포:")
display(
    issue_summary["rule_priority_hint"]
    .value_counts()
    .rename_axis("rule_priority_hint")
    .reset_index(name="issue_count")
)

print(f"최종 '상' 제한 기준: 최대 {max_high_count}개")
print("High urgency는 rule_priority_hint 계산에는 직접 사용하지 않고, priority_reason/llm_evidence_text에서 보조 지표로만 유지한다.")

검증 통과
리뷰 수: 245
리뷰-이슈 수: 477
이슈 종류 수: 17
절대 기준 후보 분포:


,priority_candidate,issue_count
0,하,10
1,중,6
2,상,1


최종 우선 검토 수준 분포:


,rule_priority_hint,issue_count
0,하,10
1,중,6
2,상,1


최종 '상' 제한 기준: 최대 5개
High urgency는 rule_priority_hint 계산에는 직접 사용하지 않고, priority_reason/llm_evidence_text에서 보조 지표로만 유지한다.


# 12. 산출 CSV 테이블 명세서

04-1에서는 04-2에서 실제로 사용할 산출물만 저장한다.

## 산출 파일 목록

| 파일명 | 역할 |
|---|---|
| `postlaunch_review_base.csv` | 리뷰 1개 단위 전처리 결과 |
| `postlaunch_issue_summary.csv` | 이슈별 반복성, 부정·혼합, Steam 비추천, 최근성, 플레이타임 근거 요약 |
| `postlaunch_patch_ops_evidence_base.csv` | 04-2 LLM 패치·운영 전략 생성용 근거 데이터 |
| `tableau_postlaunch_patch_ops_source.csv` | Tableau/보고서용 원천 데이터 |

---

## postlaunch_review_base.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `recommendationid` | Steam 리뷰 고유 ID | `221779395` |
| `appid` | Steam 게임 고유 ID | `1466060` |
| `game_name` | 게임명 | `Tainted Grail: The Fall of Avalon` |
| `review_datetime` | 리뷰 작성 일시 | `2026-04-05 16:40:12` |
| `review_recency_group` | 분석 데이터 내 최신 리뷰일 기준 최근성 구간 | `last_30d` |
| `steam_label_text` | Steam 추천/비추천 라벨 | `positive` |
| `playtime_at_review_hours` | 리뷰 작성 시점 플레이타임 | `12.5` |
| `playtime_stage` | 리뷰 작성 시점 플레이타임 구간 | `5-20h` |
| `llm_sentiment` | LLM이 리뷰 내용을 보고 분류한 감정 | `mixed` |
| `llm_urgency_candidate` | LLM이 리뷰 내용을 보고 분류한 시급도 후보 | `high` |
| `high_urgency_flag` | High urgency 여부. 우선 검토 수준 계산 기준이 아니라 보조 참고 지표 | `True` |
| `llm_review_summary` | LLM이 작성한 리뷰 요약 | `전투와 탐험은 좋지만 버그를 지적함` |
| `llm_suggested_action` | LLM이 리뷰 내용을 바탕으로 정리한 개선 방향 후보 | `진행 방해 버그를 우선 확인` |

---

## postlaunch_issue_summary.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `llm_issue_category` | LLM이 추출한 이슈 카테고리 | `gameplay_loop` |
| `issue_name_kor` | 이슈 한글명 | `게임플레이 루프` |
| `affected_review_count` | 해당 이슈가 언급된 리뷰 수 | `239` |
| `negative_mixed_review_count` |  LLM이 이슈 태그 단위에서 부정 또는 혼합 맥락으로 분류한 리뷰 수. Steam 비추천 라벨과는 별개의 LLM 분류값 | `130` |
| `steam_negative_review_count` | Steam 비추천 리뷰 중 해당 이슈가 언급된 리뷰 수 | `64` |
| `high_urgency_review_count` | LLM이 High urgency 후보로 분류한 리뷰 수. 우선 검토 수준 계산에는 직접 사용하지 않음 | `42` |
| `high_urgency_rate` | 영향 리뷰 중 High urgency 후보 비율. 보조 참고 지표 | `0.176` |
| `negative_mixed_rate` | 영향 리뷰 중 부정/혼합 비율 | `0.544` |
| `steam_negative_rate` | 영향 리뷰 중 Steam 비추천 비율 | `0.269` |
| `recent_30d_negative_mixed_review_count` | 분석 데이터 내 최신 리뷰일 기준 최근 30일 부정/혼합 리뷰 수 | `12` |
| `early_playtime_negative_mixed_review_count` | 0~5시간 구간 부정/혼합 리뷰 수 | `8` |
| `action_group_hint` | 이슈 성격과 집계값 기반 대응 구분 | `단기 개선` |
| `rule_priority_hint` | 규칙 기반 우선 검토 수준 | `상` |
| `priority_rule_detail` | 우선 검토 수준이 부여된 규칙 설명 | `부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복` |
| `priority_reason` | 우선 검토 힌트의 근거 요약 | `영향 리뷰 239개, 부정·혼합 130개...` |
| `patch_ops_note` | 패치·운영 해석 메모 | `반복 피로, 목표 구조, 보상 흐름을 점검...` |

---

## postlaunch_patch_ops_evidence_base.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `llm_issue_category` | LLM이 추출한 이슈 카테고리 | `save_progression` |
| `issue_name_kor` | 이슈 한글명 | `저장/진행` |
| `action_group_hint` | 04-1에서 계산한 대응 구분 | `즉시 확인` |
| `rule_priority_hint` | 04-1에서 계산한 고정 우선 검토 수준 | `상` |
| `priority_rule_detail` | 우선 검토 수준이 부여된 규칙 설명 | `플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락` |
| `priority_reason` | 집계 기반 근거 문장 | `영향 리뷰 26개, 부정·혼합 23개...` |
| `patch_ops_note` | 이슈별 기본 대응 메모 | `저장 손실, 진행 막힘...` |
| `llm_evidence_text` | 04-2 LLM 프롬프트에 넣을 근거 블록 | `[ISSUE] ... [/ISSUE]` |

---

## tableau_postlaunch_patch_ops_source.csv

| 컬럼명 | 설명 | 예시 값 |
|---|---|---|
| `review_datetime` | 리뷰 작성 일시 | `2026-04-05 16:40:12` |
| `review_recency_group` | 최근성 구간 | `last_30d` |
| `playtime_stage` | 플레이타임 구간 | `20-50h` |
| `llm_issue_category` | 이슈 카테고리 | `performance` |
| `issue_name_kor` | 이슈 한글명 | `성능` |
| `llm_issue_sentiment` | 해당 이슈가 리뷰에서 나타난 감정 | `negative` |
| `issue_negative_or_mixed_flag` | 부정/혼합 맥락 여부 | `True` |
| `action_group_hint` | 대응 구분 힌트 | `즉시 확인` |
| `rule_priority_hint` | 우선 검토 힌트 | `상` |
| `priority_rule_detail` | 우선 검토 힌트 부여 규칙 | `부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복` |
| `high_urgency_rate` | 이슈별 High urgency 후보 비율. 보조 참고 지표 | `0.42` |
| `patch_ops_note` | 패치·운영 해석 메모 | `성능 저하와 프레임 드랍을 우선 확인...` |
